# RTC-SDD 2027 — P01: Official Baseline Audit & Epoch 2 Continuation

**Archived, reviewed Kaggle research notebook** — source experiment completed on **19 October 2026**.

The original 34 executed code cells have been retained **in their historical order**; outputs and execution counts are removed to avoid committing verbose training logs, file paths in generated tracebacks, progress bars, or accidental secrets. This notebook documents *completed work*, not a plan to retrain. Main source-of-truth remains `src/rtcsdd/`, `scripts/`, and pinned source/configuration.

## Safety and reproducibility

- **`Run All` is non-executing by default.** It only prints skipped-cell labels. To replay a cell, explicitly add its archival ID (`S01`–`S34`) to `ENABLED_ARCHIVE_CELLS` in the control cell below. Execute prerequisites in order.
- Running historical **S27/S28 training** requires a **second independent authorization flag** `ALLOW_EPOCH2_TRAINING = True`. S26 cloud-writing requires `ALLOW_HF_WRITE = True`, and training steps require both to permit configured HF backups.
- S19/S30 (full Dev inference) require `ALLOW_GPU_INFERENCE = True`.
- The original S27 historical training command includes `--max-minutes 120`; S28 does **not**. Both experiments are finished: **do not replay** them in a fresh session unless intentionally reproducing historical work and after reviewing checkpoint locations.
- Strict security: no gated RTCFake data, access tokens, `.pt` weights, or per-ID Dev predictions are embedded in this notebook. Do not upload private Dev prediction files to public Git.
- Restoring this notebook is *not* equivalent to restoring the entire Kaggle session. See `docs/setup.md`, `docs/p01_epoch1_epoch2_audit.md`, and the private artifact manifest.

## Completed results — from archived artifact JSON files

| Result type | Epoch 1 | Epoch 2 |
|---|---:|---:|
| Optimizer step | 4,737 | 9,474 |
| Training-time weighted Dev loss | 0.670940 | 0.657102 |
| Training-time Online Dev Macro-F1 @0.5 | 0.362177 | 0.323371 |
| Saved-checkpoint Full Dev Macro-F1 @0.5 | 0.340536 | 0.317350 |
| Saved-checkpoint Online Dev Macro-F1 @0.5 | 0.360674 | 0.320433 |
| Saved-checkpoint Spoof Recall @0.5 | 0.239579 | 0.194496 |
| Saved-checkpoint ROC-AUC (Spoof positive) | 0.527068 | 0.578015 |

**Why two F1 rows?** The training-time evaluator and later re-inference of serialized checkpoints produced small differences. Do **not** splice metrics from different evaluation passes into a single comparison. Local Dev metrics are not official Codabench scores. The known P01 Epoch 1 submission (`1001718`) scored **32.2484**; teammate submission (`1001603`) scored **45.6397** and came from a different, not-yet-audited experiment. No official submission from Epoch 2 is recorded here.

**Main finding:** Epoch 2 improved score ranking (ROC-AUC +0.05095) across all examined subgroups, but its classification Macro-F1 at fixed threshold 0.5 declined. Class-weight bias is a hypothesis, *not* a proven cause. No Epoch 3 has been run.

## How to read the archive

Each archived cell has a label and a purpose. A few cells install libraries, download weights, write to private HF storage, or start actual training. Opt in **one step at a time**. This is an audit trail, not the recommended general-purpose experiment launcher.


In [ ]:
# ARCHIVE REPLAY SWITCHES — intentionally OFF by default.
# A run-all of this notebook must NEVER start training or write to HF.
ENABLED_ARCHIVE_CELLS = set()  # Example only: {"S05"} for read-only source audit.
ALLOW_EPOCH2_TRAINING = False  # Explicitly enable ONLY to deliberately replay historical training.
ALLOW_GPU_INFERENCE = False    # Explicitly enable ONLY to run full Dev inference.
ALLOW_HF_WRITE = False        # Explicitly enable ONLY when testing private HF write access/backups.
print("[SAFE MODE] All historical steps disabled by default.")


## Setup / recovery

Historical environment setup and controlled preparation; not required when inspecting archived metrics.

### S01 — 01 · Runtime and dataset inventory

Inspect Kaggle paths and attached dataset.

In [ ]:
if "S01" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S01 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import os
    import json
    import subprocess
    import textwrap

    WORKDIR = Path("/kaggle/working")
    OUR_REPO = WORKDIR / "RTC-SDD-Research"
    OFFICIAL_REPO = WORKDIR / "RTC-SDD-official"

    DATASET_ROOT = Path(
        "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
    )

    print("WORKDIR       :", WORKDIR)
    print("OUR_REPO      :", OUR_REPO)
    print("OFFICIAL_REPO :", OFFICIAL_REPO)
    print("DATASET_ROOT  :", DATASET_ROOT)
    print("DATASET EXISTS:", DATASET_ROOT.exists())

    if DATASET_ROOT.exists():
        print("\nDataset files:")
        for p in sorted(DATASET_ROOT.iterdir()):
            print(" -", p.name)


### S02 — 02 · Secret availability (no secret values displayed)

Verify Kaggle Secrets names are enabled.

In [ ]:
if "S02" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S02 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    from kaggle_secrets import UserSecretsClient

    DATASET_ROOT = Path(
        "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
    )

    assert DATASET_ROOT.is_dir(), "Dataset not found"

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    assert token, "GITHUB_TOKEN is not enabled"

    print("[PASS] RTCFake dataset found")
    print("[PASS] GitHub token accessible")
    print("[PASS] Ready to clone repository")

    del token


### S03 — 03 · Private Git repository clone and source inventory

Clone the existing private repository; no checkout overwrite.

In [ ]:
if "S03" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S03 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    from kaggle_secrets import UserSecretsClient
    import subprocess
    import tempfile
    import os

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    REPO_URL = (
        "https://github.com/AhmedLabib1/"
        "RTC-SDD-Research.git"
    )

    # Check if repository already exists
    if (ROOT / ".git").is_dir():
        print("[INFO] Repository already exists")

    elif ROOT.exists():
        raise RuntimeError(
            f"Directory exists but is not a Git repository: {ROOT}"
        )

    else:
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        assert token, "GITHUB_TOKEN is unavailable"

        with tempfile.TemporaryDirectory() as temp_dir:
            askpass = Path(temp_dir) / "askpass.sh"

            askpass.write_text(
                '#!/bin/sh\n'
                'case "$1" in\n'
                '  *Username*) printf "%s" "x-access-token" ;;\n'
                '  *Password*) printf "%s" "$GITHUB_TOKEN" ;;\n'
                'esac\n'
            )

            askpass.chmod(0o700)

            env = os.environ.copy()
            env["GITHUB_TOKEN"] = token
            env["GIT_ASKPASS"] = str(askpass)
            env["GIT_TERMINAL_PROMPT"] = "0"

            subprocess.run(
                [
                    "git",
                    "-c", "credential.helper=",
                    "clone",
                    "--depth", "1",
                    REPO_URL,
                    str(ROOT),
                ],
                env=env,
                check=True,
            )

        del token, env

        print("[PASS] Private repository cloned")

    # Verify Git commit
    result = subprocess.run(
        ["git", "-C", str(ROOT), "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    )

    print("\nCommit SHA:", result.stdout.strip())

    # Verify P01 source files
    required_files = [
        "src/rtcsdd/models/xlsr_aasist.py",
        "src/rtcsdd/training/trainer.py",
        "src/rtcsdd/training/weighted_loss.py",
        "src/rtcsdd/training/xlsr_aasist_finetune.py",
        "src/rtcsdd/eval/dev_evaluator.py",
        "src/rtcsdd/configs/p01_xlsr_aasist_official_like_fp32.json",
        "scripts/progress_inference_p01.py",
        "scripts/setup/bootstrap_official.py",
    ]

    print("\n" + "=" * 65)
    print("P01 SOURCE FILE VERIFICATION")
    print("=" * 65)

    missing = []

    for file in required_files:
        exists = (ROOT / file).is_file()

        status = "[PASS]" if exists else "[MISSING]"
        print(f"{status} {file}")

        if not exists:
            missing.append(file)

    print("\n" + "=" * 65)

    if missing:
        print(f"[WARNING] {len(missing)} required files missing")
    else:
        print("[PASS] ALL REQUIRED P01 FILES FOUND")
        print("[PASS] READY FOR OFFICIAL BASELINE AUDIT")


### S04 — 04 · Restore pinned official baseline

Bootstrap the official upstream at its recorded commit.

In [ ]:
if "S04" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S04 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import subprocess
    import sys
    import json

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    BOOTSTRAP = ROOT / "scripts/setup/bootstrap_official.py"
    PIN_FILE = ROOT / "third_party/official_rtc_sdd.pin.json"
    OFFICIAL = ROOT / "third_party/official_rtc_sdd"

    assert BOOTSTRAP.is_file()
    assert PIN_FILE.is_file()

    print("=" * 65)
    print("STEP 3 — OFFICIAL BASELINE RESTORE")
    print("=" * 65)

    # Read pinned official commit
    pin = json.loads(PIN_FILE.read_text(encoding="utf-8"))

    print("\nExpected official commit:", pin["commit_sha"])

    # Clone and verify official source
    subprocess.run(
        [sys.executable, str(BOOTSTRAP)],
        cwd=ROOT,
        check=True,
    )

    # Verify again without changing anything
    subprocess.run(
        [sys.executable, str(BOOTSTRAP), "--verify-only"],
        cwd=ROOT,
        check=True,
    )

    # Show actual Git revision
    actual_commit = subprocess.check_output(
        ["git", "-C", str(OFFICIAL), "rev-parse", "HEAD"],
        text=True,
    ).strip()

    assert actual_commit == pin["commit_sha"]

    print("\n" + "=" * 65)
    print("FINAL VERIFICATION")
    print("=" * 65)

    print("[PASS] Official repository restored")
    print("[PASS] Correct pinned commit")
    print("[PASS] Official source verified")
    print("[PASS] No model training performed")

    print("\nOfficial path:", OFFICIAL)
    print("Official commit:", actual_commit)


## Implementation and research audits

Source parity, runtime checks, subgroup and paired prediction diagnostics.

### S05 — 05 · Training loss and optimizer source audit

Compare weighted CE and optimizer implementations, read-only.

In [ ]:
if "S05" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S05 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import json
    import re

    ROOT = Path("/kaggle/working/RTC-SDD-Research")
    OFFICIAL = ROOT / "third_party/official_rtc_sdd"

    files = {
        "OFFICIAL TRAINING": (
            OFFICIAL / "xlsr_aasist/main_train.py"
        ),
        "OUR TRAINER": (
            ROOT / "src/rtcsdd/training/trainer.py"
        ),
        "OUR WEIGHTED LOSS": (
            ROOT / "src/rtcsdd/training/weighted_loss.py"
        ),
    }

    patterns = {
        "LOSS IMPLEMENTATION": (
            r"CrossEntropyLoss|cross_entropy|criterion|"
            r"weighted_ce_sum|class_weights|"
            r"loss\.backward|\.backward\("
        ),
        "OPTIMIZER IMPLEMENTATION": (
            r"torch\.optim|optim\.Adam|optimizer\s*=|"
            r"optimizer\.step|scaler\.step|"
            r"learning_rate|weight_decay|"
            r"normalize_gradients"
        ),
    }

    def show_matches(path, pattern, context=2, max_hits=15):
        lines = path.read_text(encoding="utf-8").splitlines()

        matches = [
            i for i, line in enumerate(lines)
            if re.search(pattern, line, re.IGNORECASE)
        ]

        selected = set()

        for i in matches[:max_hits]:
            selected.update(
                range(
                    max(0, i - context),
                    min(len(lines), i + context + 1)
                )
            )

        previous = -2

        for i in sorted(selected):
            if i > previous + 1:
                print("    ...")
            print(f"{i+1:04d}: {lines[i]}")
            previous = i

        print(f"\nMatched lines: {len(matches)}")

    # Verify files
    for name, path in files.items():
        assert path.is_file(), f"Missing {name}: {path}"

    print("=" * 75)
    print("STEP 4 — OFFICIAL VS P01 TRAINING AUDIT")
    print("=" * 75)

    # Compare actual implementations
    for section, pattern in patterns.items():

        print(f"\n{'#' * 75}")
        print(section)
        print(f"{'#' * 75}")

        for name, path in files.items():

            print(f"\n--- {name} ---")

            show_matches(
                path,
                pattern,
                context=2,
                max_hits=15,
            )

    # Print our exact serialized configuration
    config_path = (
        ROOT /
        "src/rtcsdd/configs/"
        "p01_xlsr_aasist_official_like_fp32.json"
    )

    config = json.loads(
        config_path.read_text(encoding="utf-8")
    )

    train_cfg = config["training"]

    print("\n" + "=" * 75)
    print("P01 SERIALIZED TRAINING CONFIG")
    print("=" * 75)

    for key in [
        "epochs",
        "batch_size",
        "gradient_accumulation",
        "optimizer",
        "learning_rate",
        "weight_decay",
        "class_weights",
        "precision",
        "gradient_clip_norm",
        "lr_scheduler",
    ]:
        print(f"{key:<28}: {train_cfg.get(key)}")

    print("\n[INFO] P01 actually completed 1 epoch.")
    print("[INFO] JSON epochs is a configured target, not completed training.")
    print("[PASS] Read-only source audit completed.")
    print("[PASS] No GPU or training required.")


### S06 — 06 · Data loading and preprocessing source audit

Inspect label mapping, crop/pad, RawBoost logic.

In [ ]:
if "S06" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S06 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import re

    ROOT = Path("/kaggle/working/RTC-SDD-Research")
    OFFICIAL = ROOT / "third_party/official_rtc_sdd"

    files = {
        "OFFICIAL DATA LOADER":
            OFFICIAL / "xlsr_aasist/utils/data_utils.py",

        "OUR PROTOCOL":
            ROOT / "src/rtcsdd/data/protocol.py",

        "OUR DATASET":
            ROOT / "src/rtcsdd/data/dataset.py",

        "OUR AUDIO PREPROCESSING":
            ROOT / "src/rtcsdd/data/audio.py",
    }

    pattern = re.compile(
        r"spoof|bonafide|label|"
        r"__getitem__|__init__|"
        r"crop|pad|repeat|tile|"
        r"64600|64_600|16000|16_000|"
        r"RawBoost|algorithm5|"
        r"return|soundfile|sf\.read",
        re.IGNORECASE,
    )

    def show_relevant_code(path, max_lines=180):
        lines = path.read_text(
            encoding="utf-8"
        ).splitlines()

        selected = set()

        for i, line in enumerate(lines):
            if pattern.search(line):
                selected.update(
                    range(
                        max(0, i - 3),
                        min(len(lines), i + 4)
                    )
                )

        selected = sorted(selected)

        print(f"Total source lines: {len(lines)}")
        print(f"Relevant lines: {len(selected)}")

        previous = -2

        for i in selected[:max_lines]:
            if i > previous + 1:
                print("    ...")

            print(f"{i+1:04d}: {lines[i]}")
            previous = i

        if len(selected) > max_lines:
            print(
                f"\n[TRUNCATED] "
                f"{len(selected) - max_lines} relevant lines omitted"
            )


    print("=" * 75)
    print("STEP 5 — DATA & PREPROCESSING AUDIT")
    print("=" * 75)

    for name, path in files.items():
        print(f"\n{'#' * 75}")
        print(name)
        print(f"{'#' * 75}")

        assert path.is_file(), f"Missing: {path}"

        show_relevant_code(path)

    print("\n[PASS] Read-only audit completed")
    print("[PASS] No GPU or training used")


### S07 — 07 · Decode/crop parity smoke test

Compare representative offline/online audio to upstream loader.

In [ ]:
if "S07" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S07 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    from io import BytesIO
    import zipfile
    import sys

    import numpy as np
    import soundfile as sf
    import librosa

    ROOT = Path("/kaggle/working/RTC-SDD-Research")
    DATASET_ROOT = Path(
        "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
    )

    sys.path.insert(0, str(ROOT / "src"))

    from rtcsdd.data.audio import crop_or_pad

    TARGET_LENGTH = 64600

    def official_pad_audio(audio, max_len=64600):
        if len(audio) >= max_len:
            return audio[:max_len]

        repeats = int(max_len / len(audio)) + 1
        return np.tile(audio, (1, repeats))[:, :max_len][0]


    print("=" * 65)
    print("STEP 5B — AUDIO DECODING PARITY")
    print("=" * 65)

    tested = 0
    failures = []

    for view in ["offline", "online"]:

        archive = DATASET_ROOT / f"dev_{view}.pack"
        assert archive.is_file(), archive

        with zipfile.ZipFile(archive) as zf:

            names = sorted(
                name for name in zf.namelist()
                if name.lower().endswith(".wav")
            )

            assert names, f"No WAV files in {archive.name}"

            # First and last examples for deterministic coverage
            selected = list(dict.fromkeys(
                [names[0], names[-1]]
            ))

            for name in selected:

                audio_bytes = zf.read(name)

                # Official decoding
                official_audio, official_sr = librosa.load(
                    BytesIO(audio_bytes),
                    sr=16000,
                    mono=True,
                )

                # P01 decoding
                our_audio, our_sr = sf.read(
                    BytesIO(audio_bytes),
                    dtype="float32",
                    always_2d=False,
                )

                assert official_sr == our_sr == 16000
                assert official_audio.ndim == our_audio.ndim == 1

                official_input = official_pad_audio(
                    official_audio, TARGET_LENGTH
                )

                our_input = crop_or_pad(
                    our_audio,
                    length=TARGET_LENGTH,
                    crop="left",
                    pad="repeat",
                )

                max_error = float(np.max(
                    np.abs(
                        official_input.astype(np.float64)
                        - our_input.astype(np.float64)
                    )
                ))

                matches = np.allclose(
                    official_input,
                    our_input,
                    rtol=0,
                    atol=1e-6,
                )

                print(f"\nView       : {view}")
                print(f"File       : {Path(name).name}")
                print(f"Input shape: {our_input.shape}")
                print(f"Max error  : {max_error:.10g}")
                print(f"Parity     : {'PASS' if matches else 'FAIL'}")

                if not matches:
                    failures.append((name, max_error))

                tested += 1

    print("\n" + "=" * 65)
    print(f"Tested : {tested}")
    print(f"Passed : {tested - len(failures)}")
    print(f"Failed : {len(failures)}")

    if failures:
        print("[WARNING] Audio decoding differences found")
    else:
        print("[PASS] All sampled audio pipelines match")


### S08 — 08 · Architecture, BatchNorm, effective batch audit

Inspect model code and actual training batch size semantics.

In [ ]:
if "S08" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S08 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    from collections import Counter
    import re
    import json

    ROOT = Path("/kaggle/working/RTC-SDD-Research")
    OFFICIAL = ROOT / "third_party/official_rtc_sdd"

    paths = {
        "model": OFFICIAL / "xlsr_aasist/model/model.py",
        "official_train": OFFICIAL / "xlsr_aasist/main_train.py",
        "official_launcher": OFFICIAL / "xlsr_aasist/run_train.sh",
        "our_adapter": ROOT / "src/rtcsdd/models/xlsr_aasist.py",
        "our_trainer": ROOT / "src/rtcsdd/training/trainer.py",
        "our_loss": ROOT / "src/rtcsdd/training/weighted_loss.py",
        "config": ROOT / "src/rtcsdd/configs/p01_xlsr_aasist_official_like_fp32.json",
    }

    for name, path in paths.items():
        assert path.is_file(), f"Missing {name}: {path}"

    source = {
        name: path.read_text(encoding="utf-8")
        for name, path in paths.items()
        if name != "config"
    }

    cfg = json.loads(paths["config"].read_text(encoding="utf-8"))
    train_cfg = cfg["training"]


    def show_matches(name, patterns, max_results=20):
        lines = source[name].splitlines()
        matched = []

        for number, line in enumerate(lines, 1):
            if any(re.search(p, line) for p in patterns):
                matched.append((number, line.strip()))

        for number, line in matched[:max_results]:
            print(f"{number:04d}: {line}")

        if len(matched) > max_results:
            print(f"... {len(matched)-max_results} more matches")


    print("=" * 70)
    print("STEP 6 — ARCHITECTURE & TRAINING DYNAMICS AUDIT")
    print("=" * 70)

    # 1. Architecture origin
    print("\n[1] MODEL IMPLEMENTATION")

    adapter = source["our_adapter"]

    print(
        "Imports official model file:",
        "xlsr_aasist/model/model.py" in adapter
    )
    print(
        "Constructs upstream Model:",
        "upstream.Model(" in adapter
    )
    print(
        "Redirects pretrained checkpoint:",
        "redirect_checkpoint" in adapter
    )

    # 2. BatchNorm
    print("\n[2] OFFICIAL MODEL BATCH NORMALIZATION")

    bn_pattern = re.compile(r"nn\.BatchNorm(1d|2d|3d)\(")

    bn_layers = bn_pattern.findall(source["model"])
    counts = Counter(bn_layers)

    print("Declared BatchNorm layers:", len(bn_layers))
    print("BatchNorm1d declarations :", counts["1d"])
    print("BatchNorm2d declarations :", counts["2d"])
    print("BatchNorm3d declarations :", counts["3d"])

    show_matches(
        "model",
        [r"nn\.BatchNorm"],
        max_results=25,
    )

    # 3. Effective batch
    print("\n[3] BATCH SIZE")

    launcher = source["official_launcher"]

    match = re.search(r"--batch_size\s+(\d+)", launcher)
    assert match, "Cannot determine official launch batch size"

    official_batch = int(match.group(1))

    micro = train_cfg["batch_size"]
    accum = train_cfg["gradient_accumulation"]

    print("Official physical batch :", official_batch)
    print("P01 micro-batch         :", micro)
    print("P01 accumulation steps  :", accum)
    print("P01 effective batch     :", micro * accum)

    # 4. Training operations
    print("\n[4] OFFICIAL TRAINING OPERATIONS")

    show_matches(
        "official_train",
        [
            r"model\.train\(",
            r"zero_grad\(",
            r"backward\(",
            r"optimizer\.step\(",
            r"clip_grad_norm_",
        ],
    )

    print("\n[5] P01 TRAINING OPERATIONS")

    show_matches(
        "our_trainer",
        [
            r"model\.train\(",
            r"zero_grad\(",
            r"backward\(",
            r"normalize_gradients\(",
            r"apply_optimizer_update\(",
        ],
    )

    print("\n[6] P01 GRADIENT NORMALIZATION")

    show_matches(
        "our_loss",
        [
            r"grad\.div_",
            r"clip_grad_norm_",
            r"cross_entropy",
        ],
    )

    # 5. Summary
    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)

    print(
        "Official explicit gradient clipping:",
        "YES" if "clip_grad_norm_" in source["official_train"] else "NO"
    )

    print(
        "P01 explicit gradient clipping:",
        "YES" if "clip_grad_norm_" in source["our_loss"] else "NO"
    )

    print("P01 configured gradient clip:", train_cfg["gradient_clip_norm"])
    print("P01 configured precision    :", train_cfg["precision"])

    print("\n[PASS] Read-only source audit finished")
    print("[PASS] No GPU, model loading, or training performed")


## Artifact restoration / provenance

Private checkpoint and small JSON restoration; never commit binaries.

### S09 — 09 · Epoch 1 serialized training artifact inspection

Download small P01 config and Dev JSON, inspect metrics.

In [ ]:
if "S09" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S09 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import json

    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import hf_hub_download

    REPO_ID = "AhmedLabib1/rtc-sdd-research-checkpoints"
    PREFIX = "experiments/p01_xlsr_aasist_official_like_fp32"

    AUDIT_DIR = Path("/kaggle/working/p01_audit")
    AUDIT_DIR.mkdir(parents=True, exist_ok=True)

    token = UserSecretsClient().get_secret("HF_TOKEN")
    assert token, "Enable HF_TOKEN in Kaggle Secrets"

    print("=" * 65)
    print("STEP 7 — P01 ACTUAL TRAINING ARTIFACTS")
    print("=" * 65)

    artifacts = {}

    for name in ["config.json", "dev_epoch_001.json"]:
        path = hf_hub_download(
            repo_id=REPO_ID,
            repo_type="model",
            filename=f"{PREFIX}/{name}",
            token=token,
            local_dir=AUDIT_DIR,
        )

        artifacts[name] = json.loads(
            Path(path).read_text(encoding="utf-8")
        )

        print("[PASS] Loaded:", name)

    del token

    # Actual effective configuration
    manifest = artifacts["config.json"]
    config = manifest.get("config", manifest)
    training = config["training"]

    print("\n" + "=" * 65)
    print("ACTUAL TRAINING CONFIG")
    print("=" * 65)

    for key in [
        "epochs",
        "batch_size",
        "gradient_accumulation",
        "learning_rate",
        "weight_decay",
        "class_weights",
        "precision",
        "gradient_clip_norm",
        "seed",
    ]:
        print(f"{key:<27}: {training.get(key)}")

    print("\nConfig fingerprint:",
          manifest.get("config_fingerprint", "Not present"))

    # Actual recorded Dev evaluation
    dev = artifacts["dev_epoch_001.json"]

    print("\n" + "=" * 65)
    print("EPOCH 1 — ACTUAL DEV RESULTS")
    print("=" * 65)

    print("Completed epoch:", dev.get("epoch"))
    print("Optimizer step :", dev.get("step"))
    print("Dev loss       :", dev.get("dev_weighted_loss"))
    print("Online Macro-F1:", dev.get("dev_online_macro_f1"))

    print("\n" + "=" * 65)
    print("FULL DEV SUBGROUP RESULTS")
    print("=" * 65)

    print(json.dumps(
        dev.get("subgroups", {}),
        indent=2,
        ensure_ascii=False
    ))

    print("\n[PASS] Training artifacts inspected")
    print("[PASS] No checkpoints modified")
    print("[PASS] No GPU or training required")


### S10 — 10 · Epoch 1 checkpoint inventory on Hugging Face

List remote files, do not download model weights.

In [ ]:
if "S10" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S10 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi

    REPO_ID = "AhmedLabib1/rtc-sdd-research-checkpoints"
    PREFIX = "experiments/p01_xlsr_aasist_official_like_fp32/"

    token = UserSecretsClient().get_secret("HF_TOKEN")
    assert token, "Enable HF_TOKEN in Kaggle Secrets"

    api = HfApi(token=token)

    files = api.list_repo_files(
        repo_id=REPO_ID,
        repo_type="model",
    )

    p01_files = sorted(
        file[len(PREFIX):]
        for file in files
        if file.startswith(PREFIX)
    )

    print("=" * 65)
    print("STEP 8 — P01 ARTIFACT INVENTORY")
    print("=" * 65)

    print("\nAvailable P01 files:")

    for file in p01_files:
        print(" -", file)

    diagnostic_files = [
        file for file in p01_files
        if any(term in file.lower() for term in [
            "score", "prediction", "logit",
            "history", "train_log", "metrics"
        ])
    ]

    print("\n" + "=" * 65)
    print("POTENTIAL DIAGNOSTIC FILES")
    print("=" * 65)

    if diagnostic_files:
        for file in diagnostic_files:
            print("[FOUND]", file)
    else:
        print("[INFO] No named prediction/history artifacts found")

    print("\nTotal P01 files:", len(p01_files))
    print("[PASS] Metadata-only inspection complete")
    print("[PASS] No model checkpoints downloaded")
    print("[PASS] No GPU or training performed")

    del token


## Setup / recovery

Historical environment setup and controlled preparation; not required when inspecting archived metrics.

### S11 — 11 · Environment and disk-space preflight

Check local prerequisites and remote checkpoint sizes.

In [ ]:
if "S11" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S11 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    import shutil
    import sys

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    EXP = "p01_xlsr_aasist_official_like_fp32"
    REPO_ID = "AhmedLabib1/rtc-sdd-research-checkpoints"

    print("=" * 65)
    print("STEP 9 — DEV INFERENCE READINESS CHECK")
    print("=" * 65)

    print("\n[1] CURRENT ENVIRONMENT")

    print("Python:", sys.version.split()[0])

    venv = Path(
        "/kaggle/working/rtcsdd-baseline-py310/bin/python"
    )

    print("Python 3.10 venv exists:", venv.is_file())

    print("\n[2] LOCAL FILES")

    paths = {
        "Research repository":
            ROOT,

        "Official source":
            ROOT / "third_party/official_rtc_sdd",

        "Pretrained XLS-R":
            Path("/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"),

        "P01 best checkpoint":
            ROOT / "outputs/experiments" / EXP / "best_online_f1.pt",
    }

    for name, path in paths.items():
        print(f"{name:<23}: {'FOUND' if path.exists() else 'MISSING'}")

    print("\n[3] AVAILABLE STORAGE")

    disk = shutil.disk_usage("/kaggle/working")

    print(f"Free space: {disk.free / 1024**3:.2f} GiB")
    print(f"Total space: {disk.total / 1024**3:.2f} GiB")

    print("\n[4] HUGGING FACE CHECKPOINT SIZES")

    token = UserSecretsClient().get_secret("HF_TOKEN")

    api = HfApi(token=token)

    info = api.repo_info(
        repo_id=REPO_ID,
        repo_type="model",
        files_metadata=True,
    )

    wanted = {
        f"experiments/{EXP}/best_online_f1.pt",
        f"experiments/{EXP}/last.pt",
    }

    for file in info.siblings:
        if file.rfilename in wanted:
            size = file.size

            if size is None:
                print(file.rfilename, "size unknown")
            else:
                print(
                    Path(file.rfilename).name,
                    f": {size / 1024**3:.2f} GiB"
                )

    del token

    print("\n[PASS] Read-only preflight completed")
    print("[PASS] No checkpoints downloaded")
    print("[PASS] No GPU inference or training performed")


### S12 — 12 · Create pinned Python 3.10 environment

Install Python/venv if missing.

In [ ]:
if "S12" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S12 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import subprocess
    import shutil
    import sys

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    VENV = Path("/kaggle/working/rtcsdd-baseline-py310")
    PY310 = VENV / "bin/python"

    print("=" * 65)
    print("STEP 10 — RESTORE PYTHON 3.10")
    print("=" * 65)

    # 1. Ensure uv is available
    uv = shutil.which("uv")

    if uv is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "uv"],
            check=True
        )
        uv = shutil.which("uv")

    assert uv is not None, "uv installation not found in PATH"

    print("[PASS] uv available:", uv)

    # 2. Install Python 3.10 interpreter
    if not PY310.is_file():

        subprocess.run(
            [uv, "python", "install", "3.10"],
            check=True
        )

        # 3. Create isolated environment
        subprocess.run(
            [
                uv, "venv",
                "--python", "3.10",
                str(VENV)
            ],
            check=True
        )

        print("[PASS] Python 3.10 venv created")

    else:
        print("[INFO] Existing environment found")

    # 4. Verify interpreter version
    version = subprocess.check_output(
        [
            str(PY310), "-c",
            "import sys; print(sys.version.split()[0])"
        ],
        text=True,
    ).strip()

    assert version.startswith("3.10."), (
        f"Expected Python 3.10, got {version}"
    )

    print("\n" + "=" * 65)
    print("ENVIRONMENT VERIFICATION")
    print("=" * 65)

    print("Python version :", version)
    print("Interpreter    :", PY310)
    print("Environment    :", VENV)

    print("\n[PASS] Python 3.10 environment ready")
    print("[INFO] CUDA/Fairseq dependencies not installed yet")
    print("[PASS] No GPU or training performed")


### S13 — 13 · Install pinned CUDA / Fairseq / project environment

Set up required dependencies in isolated venv.

In [ ]:
if "S13" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S13 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import subprocess
    import shutil
    import re

    ROOT = Path("/kaggle/working/RTC-SDD-Research")
    OFFICIAL = ROOT / "third_party/official_rtc_sdd"
    PY310 = Path("/kaggle/working/rtcsdd-baseline-py310/bin/python")

    REQUIREMENTS = OFFICIAL / "requirements.txt"
    CONVERTED = Path("/kaggle/working/rtcsdd-requirements-uv.txt")

    UV = shutil.which("uv")

    assert UV is not None, "uv is missing"
    assert PY310.is_file(), "Python 3.10 venv missing"
    assert REQUIREMENTS.is_file(), "Official requirements missing"

    print("=" * 65)
    print("STEP 11 — INSTALL PINNED P01 DEPENDENCIES")
    print("=" * 65)

    # Verify the pinned official repository is still correct.
    subprocess.run(
        [
            str(PY310),
            "-c",
            "import sys; assert sys.version_info[:2] == (3, 10)"
        ],
        check=True,
    )

    subprocess.run(
        [
            "git", "-C", str(OFFICIAL),
            "status", "--porcelain"
        ],
        check=True,
    )

    # Convert only the Fairseq requirement syntax.
    converted = []
    fairseq_count = 0

    for line in REQUIREMENTS.read_text(
        encoding="utf-8"
    ).splitlines():

        match = re.fullmatch(
            r"-e\s+git\+https://github\.com/pytorch/fairseq@"
            r"([0-9a-fA-F]{40})#egg=fairseq",
            line.strip(),
        )

        if match:
            converted.append(
                "fairseq @ git+https://github.com/"
                f"pytorch/fairseq@{match.group(1)}"
            )
            fairseq_count += 1
        else:
            converted.append(line)

    assert fairseq_count == 1, (
        f"Expected one pinned Fairseq entry, got {fairseq_count}"
    )

    for requirement in [
        "torch==2.5.1+cu121",
        "torchaudio==2.5.1+cu121",
        "torchvision==0.20.1+cu121",
    ]:
        assert requirement in converted, (
            f"Missing official pin: {requirement}"
        )

    CONVERTED.write_text(
        "\n".join(converted) + "\n",
        encoding="utf-8",
    )

    print("[PASS] Official requirements validated")
    print("[PASS] Fairseq revision preserved")
    print("[PASS] Temporary uv requirements created")

    # Install the pinned software stack.
    print("\nInstalling pinned dependencies...")

    subprocess.run(
        [
            UV, "pip", "install",
            "--python", str(PY310),
            "--index-strategy", "unsafe-best-match",
            "-r", str(CONVERTED),
            "--extra-index-url",
            "https://download.pytorch.org/whl/cu121",
        ],
        check=True,
    )

    # Install our existing project and HF checkpoint client.
    subprocess.run(
        [
            UV, "pip", "install",
            "--python", str(PY310),
            "-e", str(ROOT),
            "huggingface_hub>=0.30,<1.0",
        ],
        check=True,
    )

    print("\n" + "=" * 65)
    print("ENVIRONMENT VERIFICATION")
    print("=" * 65)

    verification = r"""
    import sys
    import torch
    import numpy as np

    # Legacy aliases used by our model adapter.
    for name, value in {
        "float": float,
        "int": int,
        "complex": complex,
        "bool": bool,
        "object": object,
        "str": str,
    }.items():
        if name not in np.__dict__:
            setattr(np, name, value)

    import fairseq
    import rtcsdd
    import soundfile

    print("Python    :", sys.version.split()[0])
    print("PyTorch   :", torch.__version__)
    print("Fairseq   : IMPORT OK")
    print("RTCS-DD   : IMPORT OK")
    print("SoundFile : IMPORT OK")
    print("CUDA built:", torch.version.cuda)

    assert torch.__version__ == "2.5.1+cu121"
    assert torch.version.cuda == "12.1"

    print("[PASS] PINNED ENVIRONMENT READY")
    """

    subprocess.run(
        [str(PY310), "-c", verification],
        check=True,
    )

    print("\n[PASS] Dependency installation completed")
    print("[PASS] No model training performed")
    print("[PASS] Existing checkpoints untouched")


## Artifact restoration / provenance

Private checkpoint and small JSON restoration; never commit binaries.

### S14 — 14 · Restore original pretrained XLS-R

Download original pretrained checkpoint and calculate SHA256.

In [ ]:
if "S14" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S14 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import hashlib
    import shutil
    import requests
    from tqdm.auto import tqdm

    PRETRAINED = Path(
        "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
    )

    URL = (
        "https://dl.fbaipublicfiles.com/"
        "fairseq/wav2vec/xlsr2_300m.pt"
    )

    PRETRAINED.parent.mkdir(parents=True, exist_ok=True)

    print("=" * 65)
    print("STEP 12 — RESTORE PRETRAINED XLS-R 300M")
    print("=" * 65)

    MIN_SIZE = 2_000_000_000

    if PRETRAINED.is_file():
        assert PRETRAINED.stat().st_size > MIN_SIZE, (
            "Existing checkpoint is unexpectedly small. "
            "Do not overwrite it without investigation."
        )
        print("[INFO] Existing pretrained checkpoint found")

    else:
        free = shutil.disk_usage("/kaggle/working").free
        print(f"Free disk: {free / 1024**3:.2f} GiB")

        assert free > 5 * 1024**3, (
            "Insufficient disk space for pretrained checkpoint"
        )

        temp = PRETRAINED.with_suffix(".pt.part")

        with requests.get(
            URL,
            stream=True,
            timeout=(30, 180),
        ) as response:

            response.raise_for_status()

            expected = int(
                response.headers.get("content-length", 0)
            )

            downloaded = 0

            with temp.open("wb") as output:
                with tqdm(
                    total=expected if expected else None,
                    unit="B",
                    unit_scale=True,
                    desc="Downloading XLS-R 300M",
                ) as progress:

                    for chunk in response.iter_content(
                        chunk_size=4 * 1024 * 1024
                    ):
                        if chunk:
                            output.write(chunk)
                            downloaded += len(chunk)
                            progress.update(len(chunk))

        assert downloaded > MIN_SIZE, (
            "Downloaded file is unexpectedly small"
        )

        if expected:
            assert downloaded == expected, (
                f"Incomplete download: {downloaded} / {expected}"
            )

        temp.replace(PRETRAINED)

        print("[PASS] Pretrained checkpoint downloaded")

    # Calculate checksum
    sha256 = hashlib.sha256()

    with PRETRAINED.open("rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024), b""
        ):
            sha256.update(chunk)

    print("\n" + "=" * 65)
    print("CHECKPOINT VERIFICATION")
    print("=" * 65)

    print("Path   :", PRETRAINED)
    print("Size   :", f"{PRETRAINED.stat().st_size / 1024**3:.2f} GiB")
    print("SHA256 :", sha256.hexdigest())

    print("\n[PASS] Pretrained XLS-R file available")
    print("[PASS] Checksum calculated")
    print("[PASS] No training performed")


### S15 — 15 · Restore Epoch 1 best model snapshot

Download and checksum the inference checkpoint.

In [ ]:
if "S15" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S15 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import sys
    import json

    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import hf_hub_download

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    sys.path.insert(0, str(ROOT / "src"))

    from rtcsdd.storage.huggingface import (
        HuggingFaceCheckpointStore,
        sha256_file,
    )

    REPO_ID = "AhmedLabib1/rtc-sdd-research-checkpoints"
    EXP = "p01_xlsr_aasist_official_like_fp32"

    REMOTE_PATH = f"experiments/{EXP}/best_online_f1.pt"

    BEST_CHECKPOINT = (
        ROOT / "outputs/experiments" /
        EXP / "best_online_f1.pt"
    )

    PRETRAINED = Path(
        "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
    )

    print("=" * 65)
    print("STEP 13 — RESTORE P01 BEST CHECKPOINT")
    print("=" * 65)

    assert PRETRAINED.is_file(), "Pretrained XLS-R missing"

    token = UserSecretsClient().get_secret("HF_TOKEN")
    assert token, "Enable HF_TOKEN in Kaggle Secrets"

    store = HuggingFaceCheckpointStore(
        REPO_ID,
        token=token,
        create_private=False,
    )

    # Verify that the checkpoint repository is private.
    store.ensure_private_repo()

    BEST_CHECKPOINT.parent.mkdir(
        parents=True, exist_ok=True
    )

    # Download only when not already present.
    if BEST_CHECKPOINT.exists():
        print("[INFO] Local checkpoint already exists")
    else:
        print("[INFO] Downloading P01 best checkpoint...")

        store.download_checkpoint(
            REMOTE_PATH,
            BEST_CHECKPOINT,
            overwrite=False,
        )

        print("[PASS] Checkpoint downloaded")

    # Retrieve its original integrity receipt.
    receipt_path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="model",
        filename=REMOTE_PATH + ".sha256.json",
        token=token,
    )

    receipt = json.loads(
        Path(receipt_path).read_text(encoding="utf-8")
    )

    assert receipt["repo_id"] == REPO_ID
    assert receipt["remote_path"] == REMOTE_PATH

    # Verify exact file size and checksum.
    actual_size = BEST_CHECKPOINT.stat().st_size
    actual_sha256 = sha256_file(BEST_CHECKPOINT)

    assert actual_size == receipt["size_bytes"], (
        "Checkpoint size mismatch"
    )

    assert actual_sha256 == receipt["sha256"], (
        "Checkpoint SHA256 mismatch"
    )

    del token

    print("\n" + "=" * 65)
    print("CHECKPOINT INTEGRITY VERIFICATION")
    print("=" * 65)

    print("File    :", BEST_CHECKPOINT.name)
    print("Size    :", f"{actual_size / 1024**3:.2f} GiB")
    print("SHA256  :", actual_sha256)

    print("\n[PASS] Private checkpoint restored")
    print("[PASS] File size verified")
    print("[PASS] SHA256 matches remote receipt")
    print("[PASS] Original pretrained XLS-R available")
    print("[PASS] No checkpoint modifications")
    print("[PASS] No training performed")


## Implementation and research audits

Source parity, runtime checks, subgroup and paired prediction diagnostics.

### S16 — 16 · Inspect Epoch 1 checkpoint metadata

Read-only checkpoint format/lineage integrity check.

In [ ]:
if "S16" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S16 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import subprocess

    ROOT = Path("/kaggle/working/RTC-SDD-Research")
    PY310 = Path(
        "/kaggle/working/rtcsdd-baseline-py310/bin/python"
    )

    EXP = "p01_xlsr_aasist_official_like_fp32"

    CHECKPOINT = (
        ROOT / "outputs/experiments" / EXP
        / "best_online_f1.pt"
    )

    ARTIFACT_DIR = (
        Path("/kaggle/working/p01_audit")
        / "experiments" / EXP
    )

    CONFIG = ARTIFACT_DIR / "config.json"
    DEV = ARTIFACT_DIR / "dev_epoch_001.json"

    for path in [PY310, CHECKPOINT, CONFIG, DEV]:
        assert path.is_file(), f"Missing: {path}"

    script = r"""
    import sys
    import json
    import math
    from collections import Counter
    from pathlib import Path

    import torch

    root = Path(sys.argv[1])
    checkpoint_path = Path(sys.argv[2])
    config_path = Path(sys.argv[3])
    dev_path = Path(sys.argv[4])

    sys.path.insert(0, str(root / "src"))
    from rtcsdd.training.config import fingerprint

    print("=" * 65)
    print("STEP 14 — P01 CHECKPOINT INSPECTION")
    print("=" * 65)

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )

    manifest = json.loads(config_path.read_text())
    config = manifest.get("config", manifest)
    dev = json.loads(dev_path.read_text())

    print("\n[1] CHECKPOINT METADATA")

    fields = [
        "format_version",
        "epoch",
        "global_step",
        "dev_loss",
        "dev_online_macro_f1",
        "stored_weight_dtype",
        "selection_metric",
        "config_fingerprint",
    ]

    for field in fields:
        print(f"{field:<25}: {checkpoint.get(field)}")

    print("\n[2] EXPERIMENT CONSISTENCY")

    assert checkpoint["format_version"] == 2
    assert checkpoint["epoch"] == 1
    assert checkpoint["global_step"] == 4737

    assert checkpoint["epoch"] == dev["epoch"]
    assert checkpoint["global_step"] == dev["step"]

    assert math.isclose(
        checkpoint["dev_loss"],
        dev["dev_weighted_loss"],
        abs_tol=1e-7,
    )

    assert math.isclose(
        checkpoint["dev_online_macro_f1"],
        dev["dev_online_macro_f1"],
        abs_tol=1e-7,
    )

    computed_fingerprint = fingerprint(config)

    assert checkpoint["config_fingerprint"] == computed_fingerprint

    if "config_fingerprint" in manifest:
        assert (
            manifest["config_fingerprint"]
            == computed_fingerprint
        )

    print("[PASS] Epoch and optimizer step match")
    print("[PASS] Dev metrics match")
    print("[PASS] Configuration fingerprint matches")

    print("\n[3] MODEL STATE DICTIONARY")

    state = checkpoint["model_state_dict"]

    print("Total tensors:", len(state))

    dtypes = Counter(str(t.dtype) for t in state.values())

    for dtype, count in sorted(dtypes.items()):
        print(f"{dtype:<20}: {count}")

    assert len(state) == 674, "Unexpected state dictionary size"

    assert checkpoint["stored_weight_dtype"] == (
        "float16_to_save_disk"
    )

    assert checkpoint["selection_metric"] == (
        "max_clean_online_dev_macro_f1_not_official"
    )

    for name, tensor in state.items():
        assert isinstance(tensor, torch.Tensor)
        if tensor.is_floating_point():
            assert tensor.dtype == torch.float16

    print("\nFirst 5 tensor names:")

    for key in list(state.keys())[:5]:
        print(" -", key, tuple(state[key].shape))

    print("\n" + "=" * 65)
    print("FINAL RESULT")
    print("=" * 65)

    print("[PASS] Checkpoint metadata verified")
    print("[PASS] Model state structure inspected")
    print("[PASS] All recorded consistency checks passed")
    print("[PASS] No GPU or training performed")
    """

    subprocess.run(
        [
            str(PY310),
            "-c",
            script,
            str(ROOT),
            str(CHECKPOINT),
            str(CONFIG),
            str(DEV),
        ],
        check=True,
    )

    print("\n[PASS] STEP 14 COMPLETE")


### S17 — 17 · GPU inference readiness check

Report T4 availability and required files.

In [ ]:
if "S17" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S17 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import subprocess
    import shutil

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    PY310 = Path(
        "/kaggle/working/rtcsdd-baseline-py310/bin/python"
    )

    PRETRAINED = Path(
        "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
    )

    BEST = (
        ROOT / "outputs/experiments/"
        "p01_xlsr_aasist_official_like_fp32/"
        "best_online_f1.pt"
    )

    DATA = Path(
        "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
    )

    print("=" * 65)
    print("STEP 15 — GPU & INFERENCE READINESS")
    print("=" * 65)

    print("\n[1] REQUIRED FILES")

    required = {
        "Python 3.10": PY310,
        "Pretrained XLS-R": PRETRAINED,
        "P01 Best Checkpoint": BEST,
        "Dev Labels": DATA / "dev_label.txt",
        "Dev Offline Pack": DATA / "dev_offline.pack",
        "Dev Online Pack": DATA / "dev_online.pack",
        "Official Model Source":
            ROOT / "third_party/official_rtc_sdd/"
            "xlsr_aasist/model/model.py",
    }

    for name, path in required.items():
        status = "PASS" if path.is_file() else "MISSING"
        print(f"{name:<25}: {status}")

    assert all(p.is_file() for p in required.values())

    print("\n[2] DISK SPACE")

    disk = shutil.disk_usage("/kaggle/working")

    print(f"Free Disk: {disk.free / 1024**3:.2f} GiB")

    print("\n[3] CUDA DEVICE CHECK")

    script = """
    import torch

    print("PyTorch version :", torch.__version__)
    print("CUDA available  :", torch.cuda.is_available())
    print("CUDA devices    :", torch.cuda.device_count())

    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)

            print(f"GPU {i}:", props.name)
            print(
                "Total VRAM:",
                f"{props.total_memory / 1024**3:.2f} GiB"
            )

        print("[PASS] GPU READY FOR INFERENCE")
    else:
        print("[INFO] GPU NOT ENABLED")
    """

    subprocess.run(
        [str(PY310), "-c", script],
        check=True,
    )

    print("\n[4] SYSTEM GPU INFORMATION")

    nvidia_smi = shutil.which("nvidia-smi")

    if nvidia_smi:
        subprocess.run([
            nvidia_smi,
            "--query-gpu=name,memory.total,memory.free",
            "--format=csv",
        ], check=True)
    else:
        print("nvidia-smi not available")

    print("\n[PASS] STEP 15 COMPLETED")
    print("[INFO] No model loaded")
    print("[INFO] No inference or training performed")


## GPU inference diagnostics

Resource-intensive inference cells are opt-in.

### S18 — 18 · Strict load of Epoch 1 snapshot

Check model initialization and state_dict strictness.

In [ ]:
if "S18" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S18 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import subprocess

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    PY310 = Path(
        "/kaggle/working/rtcsdd-baseline-py310/bin/python"
    )

    OFFICIAL = ROOT / "third_party/official_rtc_sdd"

    PRETRAINED = Path(
        "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
    )

    BEST = (
        ROOT / "outputs/experiments/"
        "p01_xlsr_aasist_official_like_fp32/"
        "best_online_f1.pt"
    )

    for path in [PY310, PRETRAINED, BEST]:
        assert path.is_file(), f"Missing: {path}"

    script = r"""
    import sys
    import torch

    from rtcsdd.models.xlsr_aasist import load_xlsr_aasist

    official = sys.argv[1]
    pretrained = sys.argv[2]
    checkpoint_path = sys.argv[3]

    device = "cuda:0"

    print("=" * 70, flush=True)
    print("STEP 16 — P01 STRICT MODEL LOADING", flush=True)
    print("=" * 70, flush=True)

    assert torch.cuda.is_available(), "CUDA unavailable"

    print("\n[1] GPU", flush=True)
    print("Device:", torch.cuda.get_device_name(0), flush=True)

    # Construct the exact upstream model.
    print("\n[2] LOADING OFFICIAL ARCHITECTURE", flush=True)

    model = load_xlsr_aasist(
        official,
        pretrained,
        device,
    )

    print("[PASS] Official XLS-R + AASIST initialized", flush=True)

    # Load our actual fine-tuned weights.
    print("\n[3] LOADING P01 CHECKPOINT", flush=True)

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )

    assert checkpoint["format_version"] == 2
    assert checkpoint["epoch"] == 1
    assert checkpoint["global_step"] == 4737

    result = model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )

    assert len(result.missing_keys) == 0
    assert len(result.unexpected_keys) == 0

    model.eval()

    print("[PASS] Fine-tuned weights loaded", flush=True)
    print("[PASS] Strict loading successful", flush=True)

    print("\n[4] MODEL VERIFICATION", flush=True)

    total_params = sum(
        p.numel() for p in model.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    print(f"Total parameters    : {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"State entries       : {len(model.state_dict())}")

    print("Epoch               :", checkpoint["epoch"])
    print("Optimizer step      :", checkpoint["global_step"])
    print("Model training mode :", model.training)

    torch.cuda.synchronize()

    allocated = torch.cuda.memory_allocated(0) / 1024**3
    peak = torch.cuda.max_memory_allocated(0) / 1024**3

    print(f"GPU memory allocated: {allocated:.2f} GiB")
    print(f"GPU peak allocation : {peak:.2f} GiB")

    print("\n" + "=" * 70, flush=True)
    print("FINAL RESULT", flush=True)
    print("=" * 70, flush=True)

    print("[PASS] Exact upstream architecture loaded")
    print("[PASS] P01 checkpoint restored with strict=True")
    print("[PASS] No missing or unexpected keys")
    print("[PASS] Model in evaluation mode")
    print("[PASS] No inference or training performed")
    """

    subprocess.run(
        [
            str(PY310),
            "-c",
            script,
            str(OFFICIAL),
            str(PRETRAINED),
            str(BEST),
        ],
        cwd=ROOT,
        check=True,
    )

    print("\n[PASS] STEP 16 COMPLETE")


### S19 — 19 · Epoch 1 Full Dev inference (Step 17)

Save resumable per-ID predictions and labeled Dev diagnostics.

In [ ]:
if "S19" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S19 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    if not ALLOW_GPU_INFERENCE:
        raise RuntimeError("Full Dev GPU inference disabled: set ALLOW_GPU_INFERENCE=True")
    from pathlib import Path
    import subprocess

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    PY310 = Path(
        "/kaggle/working/rtcsdd-baseline-py310/bin/python"
    )

    DATA = Path(
        "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
    )

    PRETRAINED = Path(
        "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
    )

    BEST = (
        ROOT / "outputs/experiments/"
        "p01_xlsr_aasist_official_like_fp32/"
        "best_online_f1.pt"
    )

    for path in [PY310, PRETRAINED, BEST, DATA / "dev_label.txt"]:
        assert path.is_file(), f"Missing: {path}"

    script = r"""
    import sys
    import json
    import math
    import time
    import hashlib
    from pathlib import Path

    import numpy as np
    import torch
    from torch.utils.data import DataLoader
    from sklearn.metrics import roc_auc_score, average_precision_score

    from rtcsdd.data.protocol import read_protocol
    from rtcsdd.data.dataset import RTCFakePackedDataset
    from rtcsdd.models.xlsr_aasist import load_xlsr_aasist
    from rtcsdd.eval.metrics import evaluate_subgroups

    ROOT, DATA, PRETRAINED, BEST = map(Path, sys.argv[1:])

    DEVICE = "cuda:0"
    OUT = ROOT / "outputs/diagnostics/p01_dev"
    OUT.mkdir(parents=True, exist_ok=True)

    PARTIAL = OUT / "dev_scores_partial.txt"
    MANIFEST = OUT / "manifest.json"
    RESULTS = OUT / "dev_diagnostics.json"

    config_path = (
        ROOT / "src/rtcsdd/configs/"
        "p01_xlsr_aasist_official_like_fp32.json"
    )

    config = json.loads(config_path.read_text())
    dcfg = config["data"]
    tcfg = config["training"]

    def sha256_file(path):
        h = hashlib.sha256()
        with path.open("rb") as f:
            for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()

    print("=" * 70, flush=True)
    print("STEP 17 — P01 FULL DEV INFERENCE", flush=True)
    print("=" * 70, flush=True)

    assert torch.cuda.is_available()
    assert tcfg["precision"] == "fp32"

    torch.manual_seed(tcfg["seed"])
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    examples = read_protocol(DATA / "dev_label.txt", "dev")
    assert len(examples) == 15204

    checkpoint_hash = sha256_file(BEST)
    protocol_hash = sha256_file(DATA / "dev_label.txt")

    identity = {
        "checkpoint_sha256": checkpoint_hash,
        "protocol_sha256": protocol_hash,
        "precision": "fp32",
        "batch_size": tcfg["eval_batch_size"],
    }

    # Protect against reusing scores from another checkpoint.
    if MANIFEST.exists():
        saved = json.loads(MANIFEST.read_text())
        assert saved == identity, "Diagnostic provenance mismatch"
    else:
        assert not PARTIAL.exists(), (
            "Partial scores exist without provenance manifest"
        )
        MANIFEST.write_text(json.dumps(identity, indent=2))

    expected_ids = {e.utterance_id for e in examples}

    scores = {}

    if PARTIAL.exists():
        for line in PARTIAL.read_text().splitlines():
            if not line.strip():
                continue

            uid, value = line.split()
            assert uid not in scores, f"Duplicate ID: {uid}"
            score = float(value)
            assert math.isfinite(score) and 0 <= score <= 1

            scores[uid] = score

    assert set(scores).issubset(expected_ids)

    pending_examples = [
        e for e in examples
        if e.utterance_id not in scores
    ]

    print("Total Dev recordings :", len(examples), flush=True)
    print("Already completed    :", len(scores), flush=True)
    print("Remaining            :", len(pending_examples), flush=True)
    print("Batch size           :", tcfg["eval_batch_size"], flush=True)
    print("Precision            : FP32", flush=True)
    print("Augmentation         : OFF", flush=True)

    if pending_examples:

        dataset = RTCFakePackedDataset(
            pending_examples,
            DATA,
            length=dcfg["audio_length"],
            crop=dcfg["crop"],
            pad=dcfg["pad"],
            rawboost={"enabled": False},
            seed=tcfg["seed"],
        )

        loader = DataLoader(
            dataset,
            batch_size=tcfg["eval_batch_size"],
            shuffle=False,
            num_workers=tcfg["workers"],
            pin_memory=True,
            drop_last=False,
        )

        print("\nLoading P01 model...", flush=True)

        model = load_xlsr_aasist(
            ROOT / "third_party/official_rtc_sdd",
            PRETRAINED,
            DEVICE,
        )

        checkpoint = torch.load(
            BEST,
            map_location="cpu",
            weights_only=True,
        )

        assert checkpoint["epoch"] == 1
        assert checkpoint["global_step"] == 4737

        model.load_state_dict(
            checkpoint["model_state_dict"],
            strict=True,
        )

        del checkpoint
        model.eval()

        print("[PASS] Model ready", flush=True)

        started = time.monotonic()
        new_count = 0
        last_report = len(scores)

        with PARTIAL.open("a", encoding="utf-8") as output:

            with torch.inference_mode():

                for audio, labels, ids in loader:

                    audio = audio.to(
                        DEVICE,
                        non_blocking=True,
                    )

                    logits = model(audio)

                    batch_scores = (
                        torch.softmax(
                            logits.float(),
                            dim=-1,
                        )[:, 0].cpu().tolist()
                    )

                    for uid, score in zip(ids, batch_scores):

                        score = float(score)

                        assert math.isfinite(score)
                        assert 0 <= score <= 1
                        assert uid not in scores

                        scores[uid] = score
                        output.write(f"{uid} {score:.10f}\n")
                        new_count += 1

                    output.flush()

                    completed = len(scores)

                    if (
                        completed - last_report >= 500
                        or completed == len(examples)
                    ):
                        elapsed = time.monotonic() - started

                        print(
                            f"[DEV] {completed}/{len(examples)} "
                            f"({completed / len(examples) * 100:.1f}%) "
                            f"elapsed={elapsed / 60:.1f} min",
                            flush=True,
                        )

                        last_report = completed

        del model
        torch.cuda.empty_cache()

    assert len(scores) == len(examples)
    assert set(scores) == expected_ids

    print("\n[PASS] All Dev scores available", flush=True)

    # Reuse the project's original evaluation logic.
    metrics = evaluate_subgroups(examples, scores)

    labels = np.array(
        [1 if e.label == 0 else 0 for e in examples]
    )

    probabilities = np.array(
        [scores[e.utterance_id] for e in examples]
    )

    roc_auc = float(
        roc_auc_score(labels, probabilities)
    )

    ap = float(
        average_precision_score(labels, probabilities)
    )

    print("\n" + "=" * 70)
    print("DEV DIAGNOSTIC RESULTS")
    print("=" * 70)

    print("Overall Macro-F1 :", metrics["all_dev"]["macro_f1"])
    print("Online Macro-F1  :", metrics["view/online"]["macro_f1"])
    print("Offline Macro-F1 :", metrics["view/offline"]["macro_f1"])
    print("Spoof Recall     :", metrics["all_dev"]["spoof"]["recall"])
    print("ROC-AUC          :", roc_auc)
    print("Average Precision:", ap)

    print("\nSCORE DISTRIBUTIONS")

    for label_id, name in [(0, "Spoof"), (1, "Bonafide")]:

        values = np.array([
            scores[e.utterance_id]
            for e in examples
            if e.label == label_id
        ])

        print(
            f"{name:<10} "
            f"count={len(values):5d} "
            f"mean={values.mean():.5f} "
            f"median={np.median(values):.5f} "
            f"p05={np.percentile(values, 5):.5f} "
            f"p95={np.percentile(values, 95):.5f}"
        )

    results = {
        "source": "P01 best_online_f1.pt",
        "metric_kind": "labeled_dev_only_not_official_score",
        "epoch": 1,
        "threshold": 0.5,
        "roc_auc": roc_auc,
        "average_precision": ap,
        "subgroups": metrics,
        "checkpoint_sha256": checkpoint_hash,
    }

    temp = RESULTS.with_suffix(".json.tmp")
    temp.write_text(
        json.dumps(results, indent=2, allow_nan=False)
    )
    temp.replace(RESULTS)

    print("\n[PASS] Diagnostic results saved")
    print("Scores :", PARTIAL)
    print("Metrics:", RESULTS)
    print("[PASS] No training or optimizer updates performed")
    print("[PASS] No Hugging Face uploads")
    """

    subprocess.run(
        [
            str(PY310),
            "-u",
            "-c",
            script,
            str(ROOT),
            str(DATA),
            str(PRETRAINED),
            str(BEST),
        ],
        cwd=ROOT,
        check=True,
    )

    print("\n[PASS] STEP 17 COMPLETE")


## Artifact backup

Local archives; do not commit gated predictions.

### S20 — 20 · Local backup of Epoch 1 diagnostics (Step 18)

Create an integrity-checked zip for local retention.

In [ ]:
if "S20" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S20 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    from zipfile import ZipFile, ZIP_DEFLATED
    from IPython.display import display, FileLink

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    DIAG = ROOT / "outputs/diagnostics/p01_dev"

    files = [
        "dev_scores_partial.txt",
        "dev_diagnostics.json",
        "manifest.json",
    ]

    BACKUP = Path(
        "/kaggle/working/p01_dev_diagnostics_backup.zip"
    )

    print("=" * 65)
    print("STEP 18 — BACKUP P01 DEV DIAGNOSTICS")
    print("=" * 65)

    for name in files:
        path = DIAG / name
        assert path.is_file(), f"Missing: {path}"

    with ZipFile(BACKUP, "w", ZIP_DEFLATED) as archive:
        for name in files:
            archive.write(DIAG / name, arcname=name)
            print("[SAVED]", name)

    with ZipFile(BACKUP) as archive:
        assert archive.testzip() is None
        assert set(archive.namelist()) == set(files)

    print("\n[PASS] Backup created")
    print("[PASS] ZIP integrity verified")
    print("[PASS] No GPU or training used")

    print("\nDownload the backup:")
    display(FileLink(str(BACKUP)))


## Implementation and research audits

Source parity, runtime checks, subgroup and paired prediction diagnostics.

### S21 — 21 · Epoch 1 ROC-AUC, subgroup and threshold diagnostics (Step 19)

CPU-only diagnostics on labeled Dev prediction scores.

In [ ]:
if "S21" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S21 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import subprocess

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    PY310 = Path(
        "/kaggle/working/rtcsdd-baseline-py310/bin/python"
    )

    DATA = Path(
        "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
    )

    DIAG = ROOT / "outputs/diagnostics/p01_dev"

    SCORES = DIAG / "dev_scores_partial.txt"

    assert PY310.is_file()
    assert SCORES.is_file()
    assert (DATA / "dev_label.txt").is_file()

    script = r"""
    import sys
    import numpy as np

    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        f1_score,
        recall_score,
    )

    from rtcsdd.data.protocol import read_protocol
    from rtcsdd.eval.submission import read_score_file

    DATA, SCORES = sys.argv[1:]

    examples = read_protocol(
        DATA + "/dev_label.txt", "dev"
    )

    scores = read_score_file(SCORES)

    assert len(examples) == 15204
    assert set(scores) == {
        e.utterance_id for e in examples
    }

    def arrays(records):
        # Binary diagnostics:
        # 1 = spoof, 0 = bonafide
        y = np.array(
            [int(e.label == 0) for e in records]
        )

        p = np.array(
            [scores[e.utterance_id] for e in records]
        )

        return y, p

    groups = {
        "All Dev": examples,
        "Online": [
            e for e in examples if e.view == "online"
        ],
        "Offline": [
            e for e in examples if e.view == "offline"
        ],
        "English": [
            e for e in examples if e.language == "en"
        ],
        "Chinese": [
            e for e in examples if e.language == "zh"
        ],
        "Online/EN": [
            e for e in examples
            if e.view == "online" and e.language == "en"
        ],
        "Online/ZH": [
            e for e in examples
            if e.view == "online" and e.language == "zh"
        ],
        "Offline/EN": [
            e for e in examples
            if e.view == "offline" and e.language == "en"
        ],
        "Offline/ZH": [
            e for e in examples
            if e.view == "offline" and e.language == "zh"
        ],
    }

    print("=" * 75)
    print("STEP 19 — P01 RANKING & THRESHOLD DIAGNOSTICS")
    print("=" * 75)

    print("\n[1] ROC-AUC BY SUBGROUP")
    print("-" * 75)

    print(
        f"{'Group':<14} {'Count':>7} "
        f"{'ROC-AUC':>10} {'AP':>10} {'AP baseline':>12}"
    )

    for name, records in groups.items():

        y, p = arrays(records)

        auc = roc_auc_score(y, p)
        ap = average_precision_score(y, p)
        prevalence = y.mean()

        print(
            f"{name:<14} {len(records):>7} "
            f"{auc:>10.4f} {ap:>10.4f} "
            f"{prevalence:>12.4f}"
        )

    print("\n[2] THRESHOLD ANALYSIS — ALL DEV")
    print("-" * 75)

    y, p = arrays(examples)

    print(
        f"{'Threshold':<12} "
        f"{'Macro-F1':>12} "
        f"{'Spoof Recall':>15} "
        f"{'Pred Spoof %':>15}"
    )

    for threshold in [
        0.40, 0.43, 0.45, 0.47,
        0.49, 0.50, 0.51, 0.53, 0.55,
    ]:

        pred = (p >= threshold).astype(int)

        macro_f1 = f1_score(
            y, pred, average="macro", zero_division=0
        )

        spoof_recall = recall_score(
            y, pred, zero_division=0
        )

        print(
            f"{threshold:<12.2f} "
            f"{macro_f1:>12.4f} "
            f"{spoof_recall:>15.4f} "
            f"{pred.mean() * 100:>14.2f}%"
        )

    print("\n[3] ALWAYS-SPOOF BASELINE")
    print("-" * 75)

    always_spoof = np.ones_like(y)

    baseline_f1 = f1_score(
        y,
        always_spoof,
        average="macro",
        zero_division=0,
    )

    print("Always-Spoof Macro-F1:", round(baseline_f1, 5))

    print("\n[4] DIAGNOSTIC BEST THRESHOLD GRID")
    print("-" * 75)

    # Research diagnostic only.
    # NOT a change to the official threshold.

    for name in ["All Dev", "Online"]:

        labels, probabilities = arrays(groups[name])

        grid = np.arange(0.35, 0.651, 0.005)

        results = []

        for threshold in grid:

            pred = (
                probabilities >= threshold
            ).astype(int)

            f1 = f1_score(
                labels,
                pred,
                average="macro",
                zero_division=0,
            )

            results.append((float(threshold), float(f1)))

        best_threshold, best_f1 = max(
            results, key=lambda x: x[1]
        )

        print(
            f"{name:<10} "
            f"best grid threshold={best_threshold:.3f} "
            f"Macro-F1={best_f1:.4f}"
        )

    print("\n" + "=" * 75)
    print("[PASS] CPU-only diagnostics completed")
    print("[PASS] No checkpoint modifications")
    print("[PASS] No GPU or training performed")
    print("[INFO] Threshold search is diagnostic only")
    """

    subprocess.run(
        [
            str(PY310),
            "-c",
            script,
            str(DATA),
            str(SCORES),
        ],
        cwd=ROOT,
        check=True,
    )


## Artifact restoration / provenance

Private checkpoint and small JSON restoration; never commit binaries.

### S22 — 22 · Restore full optimizer/resume checkpoint (Step 20)

Restore private last.pt, verify size/checksum.

In [ ]:
if "S22" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S22 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import sys
    import json
    import shutil

    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import hf_hub_download

    ROOT = Path("/kaggle/working/RTC-SDD-Research")
    sys.path.insert(0, str(ROOT / "src"))

    from rtcsdd.storage.huggingface import (
        HuggingFaceCheckpointStore,
        sha256_file,
    )

    REPO_ID = "AhmedLabib1/rtc-sdd-research-checkpoints"

    P01 = "p01_xlsr_aasist_official_like_fp32"

    REMOTE = f"experiments/{P01}/last.pt"

    # Separate continuation directory: preserve original P01.
    CONTINUATION = (
        ROOT / "outputs/experiments/p01_epoch2_continuation"
    )

    DEST = CONTINUATION / "last.pt"

    ORIGINAL_BEST = (
        ROOT / "outputs/experiments" /
        P01 / "best_online_f1.pt"
    )

    assert ORIGINAL_BEST.is_file()

    CONTINUATION.mkdir(parents=True, exist_ok=True)

    token = UserSecretsClient().get_secret("HF_TOKEN")
    assert token, "HF_TOKEN unavailable"

    print("=" * 65)
    print("STEP 20 — RESTORE P01 RESUME CHECKPOINT")
    print("=" * 65)

    # Read remote integrity receipt.
    receipt_file = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="model",
        filename=REMOTE + ".sha256.json",
        token=token,
    )

    receipt = json.loads(
        Path(receipt_file).read_text(encoding="utf-8")
    )

    assert receipt["repo_id"] == REPO_ID
    assert receipt["remote_path"] == REMOTE

    print(
        "Remote checkpoint size:",
        f"{receipt['size_bytes'] / 1024**3:.2f} GiB"
    )

    if not DEST.exists():

        free = shutil.disk_usage("/kaggle/working").free

        # Reserve room for the download and a future
        # atomic checkpoint save, plus extra headroom.
        required = (
            2.2 * receipt["size_bytes"]
            + 1.5 * 1024**3
        )

        print(f"Free disk: {free / 1024**3:.2f} GiB")

        assert free > required, (
            "Not enough disk headroom. "
            "Stop before downloading."
        )

        store = HuggingFaceCheckpointStore(
            REPO_ID,
            token=token,
            create_private=False,
        )

        store.download_checkpoint(
            REMOTE,
            DEST,
            overwrite=False,
        )

    # Verify existing or newly downloaded checkpoint.
    assert DEST.stat().st_size == receipt["size_bytes"]

    actual_sha = sha256_file(DEST)

    assert actual_sha == receipt["sha256"], (
        "Resume checkpoint SHA256 mismatch"
    )

    del token

    print("\n" + "=" * 65)
    print("RESUME CHECKPOINT VERIFICATION")
    print("=" * 65)

    print("Path  :", DEST)
    print("Size  :", f"{DEST.stat().st_size / 1024**3:.2f} GiB")
    print("SHA256:", actual_sha)

    print("\n[PASS] Resume checkpoint verified")
    print("[PASS] Original P01 checkpoint untouched")
    print("[PASS] No GPU training performed")


## Implementation and research audits

Source parity, runtime checks, subgroup and paired prediction diagnostics.

### S23 — 23 · Resume checkpoint state audit (Step 21)

Verify Adam, RNG, step and config fingerprint.

In [ ]:
if "S23" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S23 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import subprocess
    import shutil

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    PY310 = Path(
        "/kaggle/working/rtcsdd-baseline-py310/bin/python"
    )

    EXP = "p01_xlsr_aasist_official_like_fp32"

    RESUME = (
        ROOT / "outputs/experiments/"
        "p01_epoch2_continuation/last.pt"
    )

    MANIFEST = (
        Path("/kaggle/working/p01_audit")
        / "experiments" / EXP / "config.json"
    )

    CONFIG = (
        ROOT / "src/rtcsdd/configs/"
        "p01_xlsr_aasist_official_like_fp32.json"
    )

    for path in [PY310, RESUME, MANIFEST, CONFIG]:
        assert path.is_file(), f"Missing: {path}"

    script = r"""
    import sys
    import json
    import math
    import shutil
    from pathlib import Path

    import torch

    from rtcsdd.training.config import fingerprint

    resume, manifest_path, config_path = map(
        Path, sys.argv[1:]
    )

    print("=" * 65)
    print("STEP 21 — P01 RESUME STATE AUDIT")
    print("=" * 65)

    # Trusted private checkpoint, SHA256 verified in Step 20.
    # Full optimizer/RNG checkpoints may require pickle loading.
    ckpt = torch.load(
        resume,
        map_location="cpu",
        weights_only=False,
    )

    manifest = json.loads(manifest_path.read_text())
    effective_config = manifest.get("config", manifest)

    source_config = json.loads(config_path.read_text())

    expected_hash = fingerprint(effective_config)

    print("\n[1] RESUME POSITION")

    for key in [
        "format_version",
        "epoch",
        "next_batch",
        "global_step",
        "accumulated_samples",
        "best_dev_loss",
        "best_online_macro_f1",
        "no_improve_epochs",
    ]:
        print(f"{key:<25}: {ckpt.get(key)}")

    assert ckpt["format_version"] == 2
    assert ckpt["epoch"] == 1
    assert ckpt["next_batch"] == 0
    assert ckpt["global_step"] == 4737
    assert ckpt["accumulated_samples"] == 0

    print("[PASS] Ready to start Epoch 2 from batch 0")

    print("\n[2] CONFIGURATION IDENTITY")

    assert ckpt["config_fingerprint"] == expected_hash

    assert fingerprint(source_config) == expected_hash

    assert manifest["config_fingerprint"] == expected_hash

    print("[PASS] Checkpoint fingerprint matches")
    print("[PASS] Source config matches effective config")

    print("\n[3] OPTIMIZER & RNG STATE")

    optimizer = ckpt["optimizer"]

    assert "state" in optimizer
    assert "param_groups" in optimizer
    assert len(optimizer["state"]) > 0

    assert "model" in ckpt
    assert "scaler" in ckpt
    assert "rng" in ckpt

    print("Model state entries :", len(ckpt["model"]))
    print("Optimizer states    :", len(optimizer["state"]))
    print("Optimizer groups    :", len(optimizer["param_groups"]))
    print("RNG state type      :", type(ckpt["rng"]).__name__)

    assert len(ckpt["model"]) == 674

    print("[PASS] Model state available")
    print("[PASS] Adam optimizer state available")
    print("[PASS] RNG state available")

    print("\n[4] STORAGE HEADROOM")

    disk = shutil.disk_usage("/kaggle/working")

    free_gib = disk.free / 1024**3

    estimated_save_gib = (
        resume.stat().st_size * 1.05
        + 256 * 1024**2
    ) / 1024**3

    print(f"Free disk       : {free_gib:.2f} GiB")
    print(f"Save requirement: ~{estimated_save_gib:.2f} GiB")

    if free_gib < estimated_save_gib + 1.5:
        print("[WARNING] Limited space for training artifacts")
    else:
        print("[PASS] Basic checkpoint headroom available")

    print("\n[5] ORIGINAL MANIFEST PATHS")

    print("Dataset root :", manifest.get("dataset_root"))
    print("Pretrained   :", manifest.get("pretrained"))
    print("Official root:", manifest.get("official_root"))

    print("\n" + "=" * 65)
    print("[PASS] Read-only audit completed")
    print("[PASS] No checkpoints modified")
    print("[PASS] No optimizer updates or training")
    """

    subprocess.run(
        [
            str(PY310),
            "-u",
            "-c",
            script,
            str(RESUME),
            str(MANIFEST),
            str(CONFIG),
        ],
        cwd=ROOT,
        check=True,
    )


## Setup / recovery

Historical environment setup and controlled preparation; not required when inspecting archived metrics.

### S24 — 24 · Prepare continuation manifest (Step 22)

Copy small metadata, prepare output folder without training.

In [ ]:
if "S24" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S24 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import json
    import shutil
    import subprocess

    ROOT = Path("/kaggle/working/RTC-SDD-Research")
    PY310 = Path("/kaggle/working/rtcsdd-baseline-py310/bin/python")

    P01 = "p01_xlsr_aasist_official_like_fp32"

    SOURCE = Path("/kaggle/working/p01_audit/experiments") / P01

    OUT = ROOT / "outputs/experiments/p01_epoch2_continuation"

    CONFIG = (
        ROOT / "src/rtcsdd/configs/"
        "p01_xlsr_aasist_official_like_fp32.json"
    )

    LAST = OUT / "last.pt"

    DATA = Path(
        "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
    )

    PRETRAINED = Path(
        "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
    )

    print("=" * 65)
    print("STEP 22 — PREPARE EPOCH 2 CONTINUATION")
    print("=" * 65)

    # 1. Required files
    for path in [
        PY310,
        LAST,
        CONFIG,
        SOURCE / "config.json",
        SOURCE / "dev_epoch_001.json",
        DATA / "train_label.txt",
        PRETRAINED,
    ]:
        assert path.is_file(), f"Missing: {path}"

    # 2. Check experiment provenance
    manifest = json.loads(
        (SOURCE / "config.json").read_text(encoding="utf-8")
    )

    source_config = json.loads(
        CONFIG.read_text(encoding="utf-8")
    )

    assert manifest["dataset_root"] == str(DATA)
    assert manifest["pretrained"] == str(PRETRAINED)

    assert source_config["training"]["precision"] == "fp32"

    # Prevent accidental uploads to original P01 storage
    backup_settings = source_config.get("checkpoint_backup", {})

    assert backup_settings.get("enabled", False) is False, (
        "Automatic HF backup enabled. Stop and review!"
    )

    print("[PASS] Original experiment paths verified")
    print("[PASS] FP32 training configuration verified")
    print("[PASS] Automatic HF uploads disabled")

    # 3. Verify the Epoch 2 override preserves fingerprint
    verification = r"""
    import sys, json
    from rtcsdd.training.config import (
        read_config, with_overrides, fingerprint
    )

    config = read_config(sys.argv[1])
    manifest = json.load(open(sys.argv[2]))

    continued = with_overrides(config, epochs=2)

    assert fingerprint(continued) == manifest["config_fingerprint"]
    assert continued["training"]["epochs"] == 2

    print("[PASS] Epoch 2 override preserves training fingerprint")
    """

    subprocess.run(
        [
            str(PY310), "-c", verification,
            str(CONFIG),
            str(SOURCE / "config.json"),
        ],
        check=True,
    )

    # 4. Copy original small artifacts safely
    OUT.mkdir(parents=True, exist_ok=True)

    for filename in ["config.json", "dev_epoch_001.json"]:
        src = SOURCE / filename
        dst = OUT / filename

        if dst.exists():
            assert dst.read_bytes() == src.read_bytes(), (
                f"Existing {filename} differs from original"
            )
            print("[EXISTS]", filename)
        else:
            shutil.copy2(src, dst)
            print("[COPIED]", filename)

    # 5. Estimate storage headroom
    free = shutil.disk_usage("/kaggle/working").free

    atomic_save = (
        LAST.stat().st_size * 1.05
        + 256 * 1024**2
    )

    # Reserve for up to two model-only best snapshots
    best_reserve = 2 * 0.65 * 1024**3

    safety_reserve = 1.0 * 1024**3

    estimated_required = (
        atomic_save + best_reserve + safety_reserve
    )

    print("\n" + "=" * 65)
    print("STORAGE SAFETY")
    print("=" * 65)

    print(f"Free disk        : {free / 1024**3:.2f} GiB")
    print(f"Atomic save need : {atomic_save / 1024**3:.2f} GiB")
    print(f"Best reserve     : {best_reserve / 1024**3:.2f} GiB")
    print(f"Safety reserve   : {safety_reserve / 1024**3:.2f} GiB")
    print(f"Planned minimum  : {estimated_required / 1024**3:.2f} GiB")

    if free < estimated_required:
        print("[WARNING] Storage headroom is insufficient")
        print("[INFO] Do NOT start training yet")
    else:
        print("[PASS] Estimated storage headroom available")

    print("\n" + "=" * 65)
    print("FINAL STATUS")
    print("=" * 65)

    print("Continuation directory:", OUT)
    print("[PASS] Manifest prepared")
    print("[PASS] Original P01 preserved")
    print("[PASS] No training performed")


### S25 — 25 · Configure new HF continuation prefix (Step 23)

Validate runtime config and confirm backup destination unused.

In [ ]:
if "S25" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S25 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi

    import json
    import shutil
    import subprocess

    ROOT = Path("/kaggle/working/RTC-SDD-Research")
    PY310 = Path("/kaggle/working/rtcsdd-baseline-py310/bin/python")

    EXP = "p01_xlsr_aasist_official_like_fp32"

    OUT = ROOT / "outputs/experiments/p01_epoch2_continuation"

    SOURCE_CONFIG = (
        ROOT / "src/rtcsdd/configs" /
        f"{EXP}.json"
    )

    RUNTIME_CONFIG = OUT / "epoch2_runtime_config.json"

    MANIFEST = OUT / "config.json"
    LAST = OUT / "last.pt"

    REPO_ID = "AhmedLabib1/rtc-sdd-research-checkpoints"
    REMOTE_PREFIX = "experiments/p01_epoch2_continuation"

    print("=" * 65)
    print("STEP 23 — EPOCH 2 BACKUP CONFIGURATION")
    print("=" * 65)

    for path in [PY310, SOURCE_CONFIG, MANIFEST, LAST]:
        assert path.is_file(), f"Missing: {path}"

    # 1. Verify remote backup destination
    token = UserSecretsClient().get_secret("HF_TOKEN")
    assert token, "HF_TOKEN unavailable"

    api = HfApi(token=token)

    info = api.repo_info(
        repo_id=REPO_ID,
        repo_type="model",
    )

    assert info.private, "HF repository must be private"

    remote_files = api.list_repo_files(
        repo_id=REPO_ID,
        repo_type="model",
    )

    existing = [
        name for name in remote_files
        if name.startswith(REMOTE_PREFIX + "/")
    ]

    assert not existing, (
        "Continuation backup prefix already contains files. "
        "Review before proceeding: " + str(existing[:10])
    )

    print("[PASS] Private Hugging Face repository verified")
    print("[PASS] New continuation prefix is unused")

    del token

    # 2. Prepare runtime config without editing source
    cfg = json.loads(SOURCE_CONFIG.read_text(encoding="utf-8"))

    assert cfg["training"]["precision"] == "fp32"
    assert cfg["training"]["batch_size"] == 2
    assert cfg["training"]["gradient_accumulation"] == 8
    assert cfg["training"]["class_weights"] == [0.1, 0.9]

    cfg["experiment"] = "p01_epoch2_continuation"
    cfg["training"]["epochs"] = 2

    cfg["checkpoint_backup"].update({
        "enabled": True,
        "repo_id": REPO_ID,
        "remote_prefix": REMOTE_PREFIX,
        "upload_every_optimizer_steps": 1000,
        "upload_on_pause": True,
        "upload_after_epoch": True,
        "upload_best_after_epoch": True,
        "max_attempts": 3,
        "create_private": False,
        "delete_local_after_upload": False,
    })

    # 3. Verify fingerprint using pinned environment
    verification = r"""
    import sys
    import json
    from rtcsdd.training.config import read_config, fingerprint, validate

    cfg = json.load(open(sys.argv[1]))
    manifest = json.load(open(sys.argv[2]))

    validate(cfg)

    assert fingerprint(cfg) == manifest["config_fingerprint"]

    print("[PASS] Training fingerprint unchanged")
    print("[PASS] Runtime config validated")
    """

    OUT.mkdir(parents=True, exist_ok=True)

    serialized = json.dumps(cfg, indent=2, ensure_ascii=False) + "\n"

    if RUNTIME_CONFIG.exists():
        assert json.loads(RUNTIME_CONFIG.read_text()) == cfg, (
            "Existing runtime config differs. Stop and review."
        )
    else:
        temp = RUNTIME_CONFIG.with_suffix(".json.tmp")
        temp.write_text(serialized, encoding="utf-8")
        temp.replace(RUNTIME_CONFIG)

    subprocess.run(
        [
            str(PY310),
            "-c",
            verification,
            str(RUNTIME_CONFIG),
            str(MANIFEST),
        ],
        check=True,
    )

    # 4. Final storage check
    free_gib = shutil.disk_usage("/kaggle/working").free / 1024**3

    assert free_gib >= 6.0, (
        "Disk headroom below planned minimum. "
        "Do not start training."
    )

    print("\n" + "=" * 65)
    print("FINAL PLAN")
    print("=" * 65)

    print("Resume checkpoint :", LAST)
    print("Runtime config     :", RUNTIME_CONFIG)
    print("Target total epochs:", cfg["training"]["epochs"])
    print("Precision          :", cfg["training"]["precision"])
    print("HF backup prefix   :", REMOTE_PREFIX)
    print("Free disk          :", f"{free_gib:.2f} GiB")

    print("\n[PASS] Epoch 2 configuration prepared")
    print("[PASS] Original experiment remains unchanged")
    print("[PASS] No uploads or training performed")


## Private remote write

Writes an HF access-test file; authorization required.

### S26 — 26 · HF write-access preflight (Step 24)

Upload a tiny test file to preflight_checks/ only.

In [ ]:
if "S26" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S26 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    if not ALLOW_HF_WRITE:
        raise RuntimeError("Cloud writes disabled: explicitly set ALLOW_HF_WRITE=True")
    from pathlib import Path
    from io import BytesIO
    import json
    import shutil

    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    OUT = (
        ROOT / "outputs/experiments/"
        "p01_epoch2_continuation"
    )

    RUNTIME_CONFIG = OUT / "epoch2_runtime_config.json"
    LAST = OUT / "last.pt"
    MANIFEST = OUT / "config.json"

    REPO_ID = "AhmedLabib1/rtc-sdd-research-checkpoints"

    TEST_REMOTE = (
        "preflight_checks/"
        "p01_epoch2_continuation.json"
    )

    print("=" * 65)
    print("STEP 24 — FINAL CLOUD BACKUP PREFLIGHT")
    print("=" * 65)

    for path in [RUNTIME_CONFIG, LAST, MANIFEST]:
        assert path.is_file(), f"Missing: {path}"

    cfg = json.loads(RUNTIME_CONFIG.read_text())

    backup = cfg["checkpoint_backup"]

    assert cfg["training"]["epochs"] == 2
    assert cfg["training"]["precision"] == "fp32"
    assert backup["enabled"] is True

    assert backup["remote_prefix"] == (
        "experiments/p01_epoch2_continuation"
    )

    print("[PASS] Runtime configuration ready")

    token = UserSecretsClient().get_secret("HF_TOKEN")
    assert token, "HF_TOKEN unavailable"

    api = HfApi(token=token)

    info = api.repo_info(
        repo_id=REPO_ID,
        repo_type="model",
    )

    assert info.private is True

    print("[PASS] Hugging Face repository is private")

    # Small upload to verify actual write permission.
    payload = {
        "purpose": "epoch2_backup_write_test",
        "experiment": "p01_epoch2_continuation",
        "target_epochs": 2,
        "checkpoint_backup_enabled": True,
    }

    api.upload_file(
        path_or_fileobj=BytesIO(
            json.dumps(payload, sort_keys=True).encode()
        ),
        path_in_repo=TEST_REMOTE,
        repo_id=REPO_ID,
        repo_type="model",
        commit_message="Verify P01 Epoch 2 backup access",
    )

    paths = api.list_repo_files(
        repo_id=REPO_ID,
        repo_type="model",
    )

    assert TEST_REMOTE in paths

    print("[PASS] Hugging Face write permission confirmed")

    del token

    # Final disk check
    disk = shutil.disk_usage("/kaggle/working")

    free_gib = disk.free / 1024**3

    print("\n" + "=" * 65)
    print("FINAL STORAGE STATUS")
    print("=" * 65)

    print(f"Free disk: {free_gib:.2f} GiB")

    if free_gib < 6.27:
        print("[WARNING] Below estimated storage minimum")
    else:
        print("[PASS] Estimated storage minimum satisfied")

    print("\n" + "=" * 65)
    print("BACKUP SCHEDULE")
    print("=" * 65)

    print(
        "Upload every optimizer steps:",
        backup["upload_every_optimizer_steps"]
    )

    print("Upload on pause:", backup["upload_on_pause"])
    print("Upload after epoch:", backup["upload_after_epoch"])
    print("Upload best:", backup["upload_best_after_epoch"])

    print("\n[PASS] Cloud backup preflight complete")
    print("[PASS] No training performed")
    print("[PASS] Original P01 checkpoints untouched")


## Historical training continuation — completed

Do not accidentally rerun! Requires explicit opt-in and existing checkpoint state.

### S27 — 27 · Initial Epoch 2 run with time cap (Step 25)

HISTORICAL: launch training with --max-minutes=120; preserve original for reproducibility.

In [ ]:
if "S27" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S27 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    if not (ALLOW_EPOCH2_TRAINING and ALLOW_HF_WRITE):
        raise RuntimeError("Training requires explicit ALLOW_EPOCH2_TRAINING and ALLOW_HF_WRITE")
    from pathlib import Path
    from kaggle_secrets import UserSecretsClient
    import subprocess
    import shutil
    import json
    import os

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    PY310 = Path(
        "/kaggle/working/rtcsdd-baseline-py310/bin/python"
    )

    OUT = (
        ROOT / "outputs/experiments/"
        "p01_epoch2_continuation"
    )

    CONFIG = OUT / "epoch2_runtime_config.json"

    DATA = Path(
        "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
    )

    PRETRAINED = Path(
        "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
    )

    LOG = OUT / "epoch2_training.log"

    print("=" * 70)
    print("STEP 25 — CONTROLLED P01 EPOCH 2 CONTINUATION")
    print("=" * 70)

    # Verify required files
    for path in [
        PY310,
        CONFIG,
        OUT / "config.json",
        OUT / "last.pt",
        PRETRAINED,
        DATA / "train_label.txt",
    ]:
        assert path.is_file(), f"Missing: {path}"

    assert not (OUT / "dev_epoch_002.json").exists(), (
        "Epoch 2 results already exist. Stop and investigate."
    )

    cfg = json.loads(CONFIG.read_text(encoding="utf-8"))

    assert cfg["training"]["epochs"] == 2
    assert cfg["training"]["precision"] == "fp32"
    assert cfg["training"]["batch_size"] == 2
    assert cfg["training"]["gradient_accumulation"] == 8

    backup = cfg["checkpoint_backup"]

    assert backup["enabled"] is True
    assert backup["remote_prefix"] == (
        "experiments/p01_epoch2_continuation"
    )

    # Storage preflight
    free_gib = shutil.disk_usage("/kaggle/working").free / 1024**3

    print(f"\nFree disk: {free_gib:.2f} GiB")

    assert free_gib >= 6.27, (
        "Insufficient planned disk headroom. "
        "Do not start training."
    )

    # Secure token for child process
    token = UserSecretsClient().get_secret("HF_TOKEN")
    assert token, "HF_TOKEN unavailable"

    env = os.environ.copy()
    env["HF_TOKEN"] = token

    del token

    command = [
        str(PY310),
        "-u",
        "-m", "rtcsdd.training.xlsr_aasist_finetune",

        "--root", str(ROOT),
        "--dataset-root", str(DATA),
        "--checkpoint", str(PRETRAINED),
        "--config", str(CONFIG),
        "--out", str(OUT),

        "--resume",
        "--epochs", "2",
        "--max-minutes", "120",
    ]

    print("\nTraining plan:")
    print("Resume          : YES")
    print("Target epoch    : 2")
    print("Precision       : FP32")
    print("Time cap        : 120 minutes")
    print("HF backup       : ENABLED")
    print("Original P01    : UNTOUCHED")
    print("Log file        :", LOG)

    print("\n" + "=" * 70)
    print("STARTING TRAINING")
    print("=" * 70, flush=True)

    with LOG.open("a", encoding="utf-8") as log_file:

        process = subprocess.Popen(
            command,
            cwd=ROOT,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()

        returncode = process.wait()

    del env

    if returncode != 0:
        raise RuntimeError(
            f"Training exited with code {returncode}. "
            f"Check log: {LOG}"
        )

    print("\n" + "=" * 70)
    print("[PASS] TRAINING PROCESS EXITED SUCCESSFULLY")
    print("=" * 70)

    print("Log:", LOG)

    if (OUT / "dev_epoch_002.json").is_file():
        print("[PASS] Epoch 2 completed and evaluated")
    else:
        print("[INFO] Epoch 2 not yet evaluated")
        print("[INFO] Check logs for a graceful pause")


### S28 — 28 · Epoch 2 resume to completion (Step 26)

HISTORICAL: finish Epoch 2 without --max-minutes.

In [ ]:
if "S28" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S28 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    if not (ALLOW_EPOCH2_TRAINING and ALLOW_HF_WRITE):
        raise RuntimeError("Training requires explicit ALLOW_EPOCH2_TRAINING and ALLOW_HF_WRITE")
    from pathlib import Path
    from kaggle_secrets import UserSecretsClient
    import subprocess
    import shutil
    import json
    import os

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    PY310 = Path(
        "/kaggle/working/rtcsdd-baseline-py310/bin/python"
    )

    OUT = (
        ROOT / "outputs/experiments/"
        "p01_epoch2_continuation"
    )

    CONFIG = OUT / "epoch2_runtime_config.json"

    DATA = Path(
        "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
    )

    PRETRAINED = Path(
        "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
    )

    LOG = OUT / "epoch2_training.log"

    print("=" * 70)
    print("STEP 26 — RESUME EPOCH 2 UNTIL COMPLETION")
    print("=" * 70)

    for path in [
        PY310,
        CONFIG,
        OUT / "config.json",
        OUT / "last.pt",
        PRETRAINED,
        DATA / "train_label.txt",
    ]:
        assert path.is_file(), f"Missing: {path}"

    assert not (OUT / "dev_epoch_002.json").exists(), (
        "Epoch 2 already has evaluation results. "
        "Stop and inspect before resuming."
    )

    cfg = json.loads(CONFIG.read_text())

    assert cfg["training"]["epochs"] == 2
    assert cfg["training"]["precision"] == "fp32"
    assert cfg["checkpoint_backup"]["enabled"] is True
    assert cfg["checkpoint_backup"]["remote_prefix"] == (
        "experiments/p01_epoch2_continuation"
    )

    free_gib = (
        shutil.disk_usage("/kaggle/working").free / 1024**3
    )

    print(f"Free disk: {free_gib:.2f} GiB")

    assert free_gib >= 6.27, (
        "Insufficient storage headroom. "
        "Review disk usage before resuming."
    )

    token = UserSecretsClient().get_secret("HF_TOKEN")
    assert token, "HF_TOKEN unavailable"

    env = os.environ.copy()
    env["HF_TOKEN"] = token
    del token

    command = [
        str(PY310),
        "-u",
        "-m", "rtcsdd.training.xlsr_aasist_finetune",
        "--root", str(ROOT),
        "--dataset-root", str(DATA),
        "--checkpoint", str(PRETRAINED),
        "--config", str(CONFIG),
        "--out", str(OUT),
        "--resume",
        "--epochs", "2",
    ]

    assert "--max-minutes" not in command

    print("\nResume configuration:")
    print("Resume           : YES")
    print("Target Epoch     : 2")
    print("Time Limit       : NO APPLICATION-LEVEL LIMIT")
    print("Precision        : FP32")
    print("HF Backup        : ENABLED")
    print("Original P01     : UNTOUCHED")

    print("\nStarting continuation...\n", flush=True)

    with LOG.open("a", encoding="utf-8") as log_file:
        log_file.write("\n=== EPOCH 2 CONTINUATION ===\n")
        log_file.flush()

        process = subprocess.Popen(
            command,
            cwd=ROOT,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()

        returncode = process.wait()

    del env

    if returncode != 0:
        raise RuntimeError(
            f"Training exited with code {returncode}. "
            f"Check {LOG}"
        )

    if (OUT / "dev_epoch_002.json").is_file():
        print("\n[PASS] EPOCH 2 COMPLETED AND EVALUATED")
    else:
        print("\n[INFO] Epoch 2 evaluation not found")
        print("[INFO] Inspect the training log before proceeding")

    print("Training log:", LOG)


## Implementation and research audits

Source parity, runtime checks, subgroup and paired prediction diagnostics.

### S29 — 29 · Serialized Dev JSON comparison (Step 27)

Compare epoch reports without loading models.

In [ ]:
if "S29" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S29 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import json

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    EPOCH1 = Path(
        "/kaggle/working/p01_audit/experiments/"
        "p01_xlsr_aasist_official_like_fp32/"
        "dev_epoch_001.json"
    )

    EPOCH2 = (
        ROOT / "outputs/experiments/"
        "p01_epoch2_continuation/dev_epoch_002.json"
    )

    print("=" * 65)
    print("STEP 27 — EPOCH 1 VS EPOCH 2 EVALUATION AUDIT")
    print("=" * 65)

    for name, path in [
        ("Epoch 1", EPOCH1),
        ("Epoch 2", EPOCH2),
    ]:
        print(f"\n{'=' * 65}")
        print(name)
        print("=" * 65)

        assert path.is_file(), f"Missing: {path}"

        data = json.loads(path.read_text())

        print("Path:", path)
        print("Top-level keys:", list(data.keys()))

        for key in [
            "epoch",
            "global_step",
            "dev_loss",
            "dev_online_macro_f1",
            "online_macro_f1",
        ]:
            if key in data:
                print(f"{key:<25}: {data[key]}")

        for key in ["metrics", "subgroups", "groups"]:
            if key in data and isinstance(data[key], dict):
                print(f"\n{key}:")

                for group in [
                    "all_dev",
                    "view/online",
                    "view/offline",
                    "language/en",
                    "language/zh",
                ]:
                    if group in data[key]:
                        result = data[key][group]

                        if isinstance(result, dict):
                            print(
                                group,
                                "Macro-F1:",
                                result.get("macro_f1"),
                                "Count:",
                                result.get("count"),
                            )

    print("\n[PASS] Read-only evaluation inspection")
    print("[PASS] No GPU or training performed")
    print("[PASS] Checkpoints untouched")


## GPU inference diagnostics

Resource-intensive inference cells are opt-in.

### S30 — 30 · Epoch 2 Full Dev inference (Step 28)

Use epoch=2 best_dev_loss checkpoint, never Epoch 1 best_online_f1.

In [ ]:
if "S30" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S30 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    if not ALLOW_GPU_INFERENCE:
        raise RuntimeError("Full Dev GPU inference disabled: set ALLOW_GPU_INFERENCE=True")
    from pathlib import Path
    import subprocess

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    PY310 = Path(
        "/kaggle/working/rtcsdd-baseline-py310/bin/python"
    )

    DATA = Path(
        "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
    )

    PRETRAINED = Path(
        "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
    )

    # Epoch 2 inference checkpoint
    BEST = (
        ROOT / "outputs/experiments/"
        "p01_epoch2_continuation/"
        "best_dev_loss.pt"
    )

    for path in [
        PY310,
        PRETRAINED,
        BEST,
        DATA / "dev_label.txt",
    ]:
        assert path.is_file(), f"Missing: {path}"

    script = r"""
    import sys
    import json
    import math
    import time
    import hashlib
    from pathlib import Path

    import numpy as np
    import torch
    from torch.utils.data import DataLoader
    from sklearn.metrics import roc_auc_score, average_precision_score

    from rtcsdd.data.protocol import read_protocol
    from rtcsdd.data.dataset import RTCFakePackedDataset
    from rtcsdd.models.xlsr_aasist import load_xlsr_aasist
    from rtcsdd.eval.metrics import evaluate_subgroups

    ROOT, DATA, PRETRAINED, BEST = map(Path, sys.argv[1:])

    DEVICE = "cuda:0"

    OUT = ROOT / "outputs/diagnostics/p01_epoch2_dev"
    OUT.mkdir(parents=True, exist_ok=True)

    PARTIAL = OUT / "dev_scores_partial.txt"
    MANIFEST = OUT / "manifest.json"
    RESULTS = OUT / "dev_diagnostics.json"

    CONFIG_PATH = (
        ROOT / "src/rtcsdd/configs/"
        "p01_xlsr_aasist_official_like_fp32.json"
    )

    config = json.loads(CONFIG_PATH.read_text())
    dcfg = config["data"]
    tcfg = config["training"]


    def sha256_file(path):
        h = hashlib.sha256()

        with path.open("rb") as f:
            for chunk in iter(
                lambda: f.read(8 * 1024 * 1024),
                b"",
            ):
                h.update(chunk)

        return h.hexdigest()


    print("=" * 70, flush=True)
    print("STEP 28 — P01 EPOCH 2 FULL DEV INFERENCE", flush=True)
    print("=" * 70, flush=True)

    assert torch.cuda.is_available(), "GPU is required"
    assert tcfg["precision"] == "fp32"

    torch.manual_seed(tcfg["seed"])
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # -------------------------------------------------
    # 1. Verify Epoch 2 checkpoint identity
    # -------------------------------------------------

    print("\n[1] CHECKPOINT VERIFICATION", flush=True)

    checkpoint_hash = sha256_file(BEST)
    protocol_hash = sha256_file(DATA / "dev_label.txt")

    checkpoint = torch.load(
        BEST,
        map_location="cpu",
        weights_only=True,
    )

    epoch = checkpoint.get("epoch")
    step = checkpoint.get("global_step")

    print("Checkpoint :", BEST.name)
    print("Epoch      :", epoch)
    print("Step       :", step)
    print("SHA256     :", checkpoint_hash)

    assert epoch == 2, (
        f"Expected Epoch 2, found Epoch {epoch}"
    )

    assert step == 9474, (
        f"Expected optimizer step 9474, found {step}"
    )

    state_dict = checkpoint.get("model_state_dict")

    assert isinstance(state_dict, dict), (
        "Expected model_state_dict in inference checkpoint"
    )

    print("[PASS] Epoch 2 checkpoint identity verified")

    # -------------------------------------------------
    # 2. Dataset and prediction provenance
    # -------------------------------------------------

    examples = read_protocol(
        DATA / "dev_label.txt",
        "dev",
    )

    assert len(examples) == 15204

    identity = {
        "checkpoint_sha256": checkpoint_hash,
        "protocol_sha256": protocol_hash,
        "epoch": 2,
        "global_step": 9474,
        "precision": "fp32",
        "batch_size": tcfg["eval_batch_size"],
        "audio_length": dcfg["audio_length"],
        "crop": dcfg["crop"],
        "pad": dcfg["pad"],
    }

    if MANIFEST.exists():
        saved = json.loads(MANIFEST.read_text())

        assert saved == identity, (
            "Diagnostic provenance mismatch. "
            "Existing output belongs to another configuration."
        )
    else:
        assert not PARTIAL.exists(), (
            "Partial scores exist without manifest"
        )

        MANIFEST.write_text(
            json.dumps(identity, indent=2)
        )

    expected_ids = {
        e.utterance_id for e in examples
    }

    scores = {}

    if PARTIAL.exists():

        for line in PARTIAL.read_text().splitlines():

            if not line.strip():
                continue

            uid, value = line.split()

            assert uid not in scores, (
                f"Duplicate utterance: {uid}"
            )

            score = float(value)

            assert math.isfinite(score)
            assert 0 <= score <= 1

            scores[uid] = score

    assert set(scores).issubset(expected_ids)

    pending_examples = [
        e for e in examples
        if e.utterance_id not in scores
    ]

    print("\n[2] DATASET")
    print("Total Dev recordings :", len(examples))
    print("Already completed    :", len(scores))
    print("Remaining            :", len(pending_examples))
    print("Batch size           :", tcfg["eval_batch_size"])
    print("Precision            : FP32")
    print("Augmentation         : OFF")

    # -------------------------------------------------
    # 3. Load model and perform inference
    # -------------------------------------------------

    if pending_examples:

        dataset = RTCFakePackedDataset(
            pending_examples,
            DATA,
            length=dcfg["audio_length"],
            crop=dcfg["crop"],
            pad=dcfg["pad"],
            rawboost={"enabled": False},
            seed=tcfg["seed"],
        )

        loader = DataLoader(
            dataset,
            batch_size=tcfg["eval_batch_size"],
            shuffle=False,
            num_workers=tcfg["workers"],
            pin_memory=True,
            drop_last=False,
        )

        print("\n[3] LOADING EPOCH 2 MODEL", flush=True)

        model = load_xlsr_aasist(
            ROOT / "third_party/official_rtc_sdd",
            PRETRAINED,
            DEVICE,
        )

        model.load_state_dict(
            state_dict,
            strict=True,
        )

        del checkpoint
        del state_dict

        model.eval()

        print("[PASS] Epoch 2 weights loaded with strict=True")
        print("[PASS] Model in evaluation mode", flush=True)

        started = time.monotonic()
        last_report = len(scores)

        with PARTIAL.open("a", encoding="utf-8") as output:

            with torch.inference_mode():

                for audio, labels, ids in loader:

                    audio = audio.to(
                        DEVICE,
                        non_blocking=True,
                    )

                    logits = model(audio)

                    batch_scores = (
                        torch.softmax(
                            logits.float(),
                            dim=-1,
                        )[:, 0].cpu().tolist()
                    )

                    for uid, score in zip(ids, batch_scores):

                        score = float(score)

                        assert math.isfinite(score)
                        assert 0 <= score <= 1
                        assert uid not in scores

                        scores[uid] = score

                        output.write(
                            f"{uid} {score:.10f}\n"
                        )

                    output.flush()

                    completed = len(scores)

                    if (
                        completed - last_report >= 500
                        or completed == len(examples)
                    ):
                        elapsed = time.monotonic() - started

                        print(
                            f"[DEV] {completed}/{len(examples)} "
                            f"({completed / len(examples) * 100:.1f}%) "
                            f"elapsed={elapsed / 60:.1f} min",
                            flush=True,
                        )

                        last_report = completed

        del model
        torch.cuda.empty_cache()

    else:
        del checkpoint
        del state_dict

    # -------------------------------------------------
    # 4. Evaluate all Dev predictions
    # -------------------------------------------------

    assert len(scores) == len(examples)
    assert set(scores) == expected_ids

    print("\n[PASS] All Dev scores available", flush=True)

    metrics = evaluate_subgroups(
        examples,
        scores,
    )

    # sklearn positive class = 1 for spoof
    # Original RTCFake labels remain:
    # spoof=0, bonafide=1

    labels = np.array(
        [int(e.label == 0) for e in examples]
    )

    probabilities = np.array(
        [scores[e.utterance_id] for e in examples]
    )

    roc_auc = float(
        roc_auc_score(labels, probabilities)
    )

    ap = float(
        average_precision_score(
            labels,
            probabilities,
        )
    )

    # -------------------------------------------------
    # 5. Report results
    # -------------------------------------------------

    print("\n" + "=" * 70)
    print("EPOCH 2 — DEV DIAGNOSTIC RESULTS")
    print("=" * 70)

    print(
        "Overall Macro-F1 :",
        metrics["all_dev"]["macro_f1"],
    )

    print(
        "Online Macro-F1  :",
        metrics["view/online"]["macro_f1"],
    )

    print(
        "Offline Macro-F1 :",
        metrics["view/offline"]["macro_f1"],
    )

    print(
        "Spoof Recall     :",
        metrics["all_dev"]["spoof"]["recall"],
    )

    print("ROC-AUC          :", roc_auc)
    print("Average Precision:", ap)

    print("\nSCORE DISTRIBUTIONS")

    for label_id, name in [
        (0, "Spoof"),
        (1, "Bonafide"),
    ]:

        values = np.array([
            scores[e.utterance_id]
            for e in examples
            if e.label == label_id
        ])

        print(
            f"{name:<10} "
            f"count={len(values):5d} "
            f"mean={values.mean():.5f} "
            f"median={np.median(values):.5f} "
            f"p05={np.percentile(values, 5):.5f} "
            f"p95={np.percentile(values, 95):.5f}"
        )

    # -------------------------------------------------
    # 6. Save diagnostics
    # -------------------------------------------------

    results = {
        "source": "P01 epoch2 best_dev_loss.pt",
        "metric_kind": "labeled_dev_only_not_official_score",
        "epoch": 2,
        "global_step": 9474,
        "threshold": 0.5,
        "roc_auc": roc_auc,
        "average_precision": ap,
        "subgroups": metrics,
        "checkpoint_sha256": checkpoint_hash,
    }

    temp = RESULTS.with_suffix(".json.tmp")

    temp.write_text(
        json.dumps(
            results,
            indent=2,
            allow_nan=False,
        )
    )

    temp.replace(RESULTS)

    print("\n" + "=" * 70)
    print("[PASS] EPOCH 2 DIAGNOSTICS COMPLETE")
    print("=" * 70)

    print("Scores :", PARTIAL)
    print("Metrics:", RESULTS)

    print("[PASS] No training performed")
    print("[PASS] No checkpoint modifications")
    print("[PASS] No Hugging Face uploads")
    """

    subprocess.run(
        [
            str(PY310),
            "-u",
            "-c",
            script,
            str(ROOT),
            str(DATA),
            str(PRETRAINED),
            str(BEST),
        ],
        cwd=ROOT,
        check=True,
    )

    print("\n[PASS] STEP 28 COMPLETE")


## Implementation and research audits

Source parity, runtime checks, subgroup and paired prediction diagnostics.

### S31 — 31 · Paired Epoch 1 / Epoch 2 diagnostics (Step 29)

ROC-AUC by subgroup, prediction changes, score shifts.

In [ ]:
if "S31" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S31 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import subprocess

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    PY310 = Path(
        "/kaggle/working/rtcsdd-baseline-py310/bin/python"
    )

    DATA = Path(
        "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
    )

    E1 = ROOT / "outputs/diagnostics/p01_dev"
    E2 = ROOT / "outputs/diagnostics/p01_epoch2_dev"

    script = r"""
    import sys
    import json
    import numpy as np
    from pathlib import Path

    from sklearn.metrics import (
        roc_auc_score,
        f1_score,
        recall_score,
    )

    from rtcsdd.data.protocol import read_protocol
    from rtcsdd.eval.submission import read_score_file

    DATA, E1, E2 = map(Path, sys.argv[1:])

    print("=" * 75)
    print("STEP 29 — EPOCH 1 VS EPOCH 2 ERROR ANALYSIS")
    print("=" * 75)

    examples = read_protocol(
        DATA / "dev_label.txt", "dev"
    )

    scores1 = read_score_file(
        E1 / "dev_scores_partial.txt"
    )

    scores2 = read_score_file(
        E2 / "dev_scores_partial.txt"
    )

    ids = {e.utterance_id for e in examples}

    assert len(examples) == 15204
    assert set(scores1) == ids
    assert set(scores2) == ids

    # Verify saved prediction provenance.
    m1 = json.loads((E1 / "manifest.json").read_text())
    m2 = json.loads((E2 / "manifest.json").read_text())

    assert m1["protocol_sha256"] == m2["protocol_sha256"]
    assert m1["checkpoint_sha256"] != m2["checkpoint_sha256"]

    print("[PASS] Same Dev protocol")
    print("[PASS] Different checkpoint identities")
    print("[PASS] All 15,204 predictions available")

    def arrays(records):
        # For sklearn: positive class 1 = Spoof.
        y = np.array([
            int(e.label == 0) for e in records
        ])

        p1 = np.array([
            scores1[e.utterance_id] for e in records
        ])

        p2 = np.array([
            scores2[e.utterance_id] for e in records
        ])

        return y, p1, p2

    groups = {
        "All Dev": examples,
        "Online": [e for e in examples if e.view == "online"],
        "Offline": [e for e in examples if e.view == "offline"],
        "English": [e for e in examples if e.language == "en"],
        "Chinese": [e for e in examples if e.language == "zh"],
        "Online/EN": [
            e for e in examples
            if e.view == "online" and e.language == "en"
        ],
        "Online/ZH": [
            e for e in examples
            if e.view == "online" and e.language == "zh"
        ],
        "Offline/EN": [
            e for e in examples
            if e.view == "offline" and e.language == "en"
        ],
        "Offline/ZH": [
            e for e in examples
            if e.view == "offline" and e.language == "zh"
        ],
    }

    print("\n[1] ROC-AUC COMPARISON")
    print("-" * 75)

    print(
        f"{'Group':<15} {'Count':>7} "
        f"{'Epoch 1':>10} {'Epoch 2':>10} {'Delta':>10}"
    )

    report = {}

    for name, records in groups.items():
        y, p1, p2 = arrays(records)

        auc1 = float(roc_auc_score(y, p1))
        auc2 = float(roc_auc_score(y, p2))

        report[name] = {
            "count": len(records),
            "epoch1_auc": auc1,
            "epoch2_auc": auc2,
            "delta_auc": auc2 - auc1,
        }

        print(
            f"{name:<15} {len(records):>7} "
            f"{auc1:>10.4f} {auc2:>10.4f} "
            f"{auc2-auc1:>+10.4f}"
        )

    print("\n[2] MACRO-F1 AND SPOOF RECALL AT THRESHOLD 0.5")
    print("-" * 75)

    print(
        f"{'Metric':<24} {'Epoch 1':>12} "
        f"{'Epoch 2':>12} {'Delta':>12}"
    )

    y, p1, p2 = arrays(examples)

    assert np.isfinite(p1).all()
    assert np.isfinite(p2).all()

    pred1 = (p1 >= 0.5).astype(int)
    pred2 = (p2 >= 0.5).astype(int)

    measurements = {
        "Macro-F1": (
            f1_score(y, pred1, average="macro"),
            f1_score(y, pred2, average="macro"),
        ),
        "Spoof Recall": (
            recall_score(y, pred1),
            recall_score(y, pred2),
        ),
        "Predicted Spoof Rate": (
            pred1.mean(),
            pred2.mean(),
        ),
    }

    for name, (v1, v2) in measurements.items():
        print(
            f"{name:<24} {v1:>12.4f} "
            f"{v2:>12.4f} {v2-v1:>+12.4f}"
        )

    print("\n[3] PAIRED PREDICTION CHANGES")
    print("-" * 75)

    for label, positive in [
        ("Actual Spoof", 1),
        ("Actual Bonafide", 0),
    ]:
        mask = (y == positive)

        old_correct = pred1[mask] == positive
        new_correct = pred2[mask] == positive

        lost = int(np.sum(old_correct & ~new_correct))
        gained = int(np.sum(~old_correct & new_correct))

        print(f"\n{label}")
        print("Total:", int(mask.sum()))
        print("Correct -> Wrong:", lost)
        print("Wrong -> Correct:", gained)
        print("Net correct change:", gained - lost)

    print("\n[4] SCORE DISTRIBUTION CHANGES")
    print("-" * 75)

    for label, positive in [
        ("Spoof", 1),
        ("Bonafide", 0),
    ]:
        mask = y == positive

        print(
            f"{label:<10} "
            f"E1 mean={p1[mask].mean():.5f} "
            f"E2 mean={p2[mask].mean():.5f} "
            f"Delta={p2[mask].mean()-p1[mask].mean():+.5f}"
        )

    out = (
        Path("/kaggle/working/RTC-SDD-Research")
        / "outputs/diagnostics/p01_epoch_comparison"
    )

    out.mkdir(parents=True, exist_ok=True)

    result_file = out / "epoch1_vs_epoch2.json"

    report["summary"] = {
        "epoch1": {
            "macro_f1": float(measurements["Macro-F1"][0]),
            "spoof_recall": float(measurements["Spoof Recall"][0]),
        },
        "epoch2": {
            "macro_f1": float(measurements["Macro-F1"][1]),
            "spoof_recall": float(measurements["Spoof Recall"][1]),
        },
        "threshold": 0.5,
    }

    result_file.write_text(
        json.dumps(report, indent=2, allow_nan=False)
    )

    print("\n" + "=" * 75)
    print("[PASS] PAIRED ANALYSIS COMPLETE")
    print("[PASS] No GPU inference or training")
    print("[PASS] Existing scores and checkpoints untouched")
    print("Report:", result_file)
    """

    subprocess.run(
        [
            str(PY310),
            "-u",
            "-c",
            script,
            str(DATA),
            str(E1),
            str(E2),
        ],
        cwd=ROOT,
        check=True,
    )


## Artifact backup

Local archives; do not commit gated predictions.

### S32 — 32 · Diagnostics-only backup (early archive cell)

Historical smaller local ZIP, superseded by complete research backup.

In [ ]:
if "S32" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S32 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    from zipfile import ZipFile, ZIP_DEFLATED
    from IPython.display import display, FileLink

    ROOT = Path("/kaggle/working/RTC-SDD-Research")
    DIAG = ROOT / "outputs/diagnostics"

    FOLDERS = [
        "p01_dev",
        "p01_epoch2_dev",
        "p01_epoch_comparison",
    ]

    BACKUP = Path(
        "/kaggle/working/"
        "p01_epoch1_epoch2_diagnostics_backup.zip"
    )

    required = {
        "p01_dev": [
            "manifest.json",
            "dev_scores_partial.txt",
            "dev_diagnostics.json",
        ],
        "p01_epoch2_dev": [
            "manifest.json",
            "dev_scores_partial.txt",
            "dev_diagnostics.json",
        ],
        "p01_epoch_comparison": [
            "epoch1_vs_epoch2.json",
        ],
    }

    for folder, filenames in required.items():
        for filename in filenames:
            path = DIAG / folder / filename
            assert path.is_file(), f"Missing: {path}"

    with ZipFile(BACKUP, "w", ZIP_DEFLATED) as archive:
        for folder in FOLDERS:
            for path in sorted((DIAG / folder).rglob("*")):
                if path.is_file():
                    archive.write(
                        path,
                        arcname=path.relative_to(ROOT),
                    )

    with ZipFile(BACKUP) as archive:
        assert archive.testzip() is None
        print("Files backed up:", len(archive.namelist()))

    print("[PASS] Diagnostics backup verified")
    print("[PASS] Checkpoints untouched")
    print("[PASS] No GPU or training used")
    print("Backup:", BACKUP)

    display(FileLink(str(BACKUP)))


### S33 — 33 · Full research artifact backup (Step 30)

Create ZIP including manifests, diagnostics, training logs.

In [ ]:
if "S33" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S33 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    from zipfile import ZipFile, ZIP_DEFLATED
    from IPython.display import display, FileLink
    import hashlib
    import json

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    DIAG = ROOT / "outputs/diagnostics"

    EXP = (
        ROOT / "outputs/experiments/"
        "p01_epoch2_continuation"
    )

    BACKUP = Path(
        "/kaggle/working/"
        "p01_epoch1_epoch2_research_backup.zip"
    )

    print("=" * 70)
    print("STEP 30 — FINAL RESEARCH ARTIFACTS BACKUP")
    print("=" * 70)

    required = [
        DIAG / "p01_dev/manifest.json",
        DIAG / "p01_dev/dev_scores_partial.txt",
        DIAG / "p01_dev/dev_diagnostics.json",

        DIAG / "p01_epoch2_dev/manifest.json",
        DIAG / "p01_epoch2_dev/dev_scores_partial.txt",
        DIAG / "p01_epoch2_dev/dev_diagnostics.json",

        DIAG / "p01_epoch_comparison/epoch1_vs_epoch2.json",

        EXP / "config.json",
        EXP / "epoch2_runtime_config.json",
        EXP / "dev_epoch_001.json",
        EXP / "dev_epoch_002.json",
        EXP / "epoch2_training.log",
    ]

    missing = [p for p in required if not p.is_file()]

    if missing:
        for path in missing:
            print("[MISSING]", path)
        raise FileNotFoundError("Required research artifacts missing")

    print(f"[PASS] Found {len(required)} required files")

    manifest = {
        "experiment": "p01_epoch2_continuation",
        "completed_epochs": 2,
        "optimizer_steps": 9474,
        "checkpoint_storage": "Private Hugging Face",
        "files": [],
    }

    with ZipFile(BACKUP, "w", ZIP_DEFLATED) as zf:

        for path in required:

            rel = str(path.relative_to(ROOT))

            content = path.read_bytes()

            digest = hashlib.sha256(content).hexdigest()

            zf.writestr(rel, content)

            manifest["files"].append({
                "path": rel,
                "size_bytes": len(content),
                "sha256": digest,
            })

            print("[SAVED]", rel)

        zf.writestr(
            "backup_manifest.json",
            json.dumps(manifest, indent=2),
        )

    with ZipFile(BACKUP) as zf:
        assert zf.testzip() is None

        for entry in manifest["files"]:
            content = zf.read(entry["path"])

            assert hashlib.sha256(content).hexdigest() == (
                entry["sha256"]
            )

    print("\n" + "=" * 70)
    print("[PASS] BACKUP CREATED AND VERIFIED")
    print("=" * 70)

    print("Files:", len(required))
    print("Archive:", BACKUP)
    print("Size (MB):", round(BACKUP.stat().st_size / 1024**2, 2))

    print("\n[PASS] No training performed")
    print("[PASS] No checkpoint modifications")
    print("[PASS] No cloud uploads")

    display(FileLink(str(BACKUP)))


## Implementation and research audits

Source parity, runtime checks, subgroup and paired prediction diagnostics.

### S34 — 34 · Git working tree audit (Step 31)

Read-only summary of tracked/untracked files and ignored research outputs.

In [ ]:
if "S34" not in ENABLED_ARCHIVE_CELLS:
    print("[SKIP] S34 — archived; opt in via ENABLED_ARCHIVE_CELLS")
else:
    from pathlib import Path
    import subprocess

    ROOT = Path("/kaggle/working/RTC-SDD-Research")

    assert (ROOT / ".git").is_dir(), "Git repository not found"

    def git(*args):
        result = subprocess.run(
            ["git", "-C", str(ROOT), *args],
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()

    print("=" * 70)
    print("STEP 31 — GIT REPOSITORY AUDIT")
    print("=" * 70)

    print("\n[1] GIT IDENTITY")
    print("Branch :", git("branch", "--show-current"))
    print("Commit :", git("rev-parse", "HEAD"))
    print("Remotes:", git("remote"))

    print("\n[2] WORKING TREE STATUS")

    status = git(
        "status",
        "--short",
        "--untracked-files=all",
    )

    print(status if status else "[CLEAN] No Git changes")

    print("\n[3] TRACKED MODIFICATIONS")

    diff = git("diff", "--stat")
    print(diff if diff else "[NONE]")

    print("\n[4] NOTEBOOK INVENTORY")

    notebooks = ROOT / "notebooks"

    for path in sorted(notebooks.glob("*.ipynb")):
        tracked = bool(
            git("ls-files", "--", str(path.relative_to(ROOT)))
        )
        print(
            "[TRACKED]" if tracked else "[UNTRACKED]",
            path.name,
        )

    print("\n[5] DOCUMENTATION INVENTORY")

    docs = ROOT / "docs"

    for path in sorted(docs.rglob("*.md")):
        print("-", path.relative_to(ROOT))

    print("\n[6] RESEARCH ARTIFACT GIT STATUS")

    targets = [
        "outputs/diagnostics/p01_dev",
        "outputs/diagnostics/p01_epoch2_dev",
        "outputs/diagnostics/p01_epoch_comparison",
        "outputs/experiments/p01_epoch2_continuation",
    ]

    for target in targets:

        path = ROOT / target

        ignored = subprocess.run(
            [
                "git", "-C", str(ROOT),
                "check-ignore", "-q", target,
            ],
            check=False,
        ).returncode == 0

        print(
            target,
            "| EXISTS:", path.exists(),
            "| IGNORED:", ignored,
        )

    print("\n[7] README")

    readme = ROOT / "README.md"

    print("README exists:", readme.is_file())

    if readme.is_file():
        print("README size:", readme.stat().st_size, "bytes")

    print("\n" + "=" * 70)
    print("[PASS] READ-ONLY GIT AUDIT COMPLETE")
    print("[PASS] No files modified")
    print("[PASS] No Git commit or push")
    print("[PASS] No GPU required")


## Archive closeout — research status

**Completed:** 2 FP32 training epochs, verified HF backups, checkpoint-specific Dev inference, labeled Dev ranking/threshold/subgroup analysis, paired error comparison, private research artifact ZIP and Git inventory.

**Preserve separately:** the full optimizer `last.pt` for exact resume; Epoch 1 `best_online_f1.pt` for prior selected model; Epoch 2 `best_dev_loss.pt` for Epoch 2 diagnostics; the local backup `p01_epoch1_epoch2_research_backup.zip` for per-utterance Dev scores and logs (not in Git).

**Not completed:** Epoch 3, CE class-weight ablations, CE+PCL training attribution from teammate, other pretrained backbones. Never label these as measured results.

**Next research decision:** use existing diagnostics to plan budget-matched exploration vs. further continuation; compare with fixed 0.5 threshold and clearly distinguish clean Dev from official hidden Progress/Final metrics.
